# Hallway Illuminance Prototype From Google Drive Datasets

This notebook is fully self-contained and is designed around your Google Drive copies of:
- `NYU Depth V2` in `/content/drive/MyDrive/hallway_lighting_datasets/nyu_depth_v2/archive.zip`
- `MID` train/test JPG zips in `/content/drive/MyDrive/hallway_lighting_datasets/MID_dataset/...`

Main outputs:
- floor-plane average illuminance
- low illuminance `p5`
- high illuminance `p95`
- illuminance discrepancy `p95 - p5`
- point-wise lux directly under fixtures
- point-wise lux between adjacent fixtures
- estimated power, 5-minute interval energy, and 5-minute interval carbon

Important reality check: with only NYU + MID and no direct hallway lux labels, this is a public-datasets-only prototype. It can be useful for a prototype and demonstration, but it is not a certified lux meter and it will not guarantee calibrated absolute illuminance in every hallway. Improvements needed for production are described at the end.


## 1. Install Dependencies

Run this once per Colab session.


In [ ]:
%pip install -q torch torchvision PyYAML numpy pandas matplotlib Pillow tqdm scikit-image scipy h5py kagglehub onnx onnxruntime

## 2. Imports And Global Config

This cell contains the exact Drive zip paths and the prototype carbon parameters. If your hallway or grid settings differ, adjust the photometric parameters here.


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Mapping, Sequence
import json
import random
import shutil
import tarfile
import time
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy.io import loadmat
from skimage.io import imread
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    pass

IN_KAGGLE = Path('/kaggle/input').exists()

if IN_COLAB:
    RUN_ROOT = Path('/content/drive/MyDrive/hallway_lighting_runs').resolve()
    CACHE_ROOT = Path('/content/hallway_lighting_runtime_cache').resolve()
elif IN_KAGGLE:
    RUN_ROOT = Path('/kaggle/working/hallway_lighting_drive_prototype_runs').resolve()
    CACHE_ROOT = Path('/kaggle/working/hallway_lighting_runtime_cache').resolve()
else:
    RUN_ROOT = Path('./hallway_lighting_drive_prototype_runs').resolve()
    CACHE_ROOT = RUN_ROOT / 'runtime_cache'

MANIFEST_DIR = CACHE_ROOT / 'manifests'
PREPARED_ROOT = CACHE_ROOT / 'prepared_datasets'
CHECKPOINT_DIR = RUN_ROOT / 'checkpoints'
VIS_DIR = RUN_ROOT / 'visualizations'
INFER_DIR = RUN_ROOT / 'inference'
EXPORT_DIR = RUN_ROOT / 'exports'
REPORT_DIR = RUN_ROOT / 'reports'

def ensure_run_directories() -> None:
    for directory in (RUN_ROOT, CACHE_ROOT, MANIFEST_DIR, PREPARED_ROOT, CHECKPOINT_DIR, VIS_DIR, INFER_DIR, EXPORT_DIR, REPORT_DIR):
        directory.mkdir(parents=True, exist_ok=True)

# Exact Google Drive paths provided by the user.
NYU_DRIVE_ZIP = Path('/content/drive/MyDrive/hallway_lighting_datasets/nyu_depth_v2/archive.zip')
MID_TRAIN_DRIVE_ZIP = Path('/content/drive/MyDrive/hallway_lighting_datasets/MID_dataset/multi_illumination_train_mip2_jpg.zip')
MID_TEST_DRIVE_ZIP = Path('/content/drive/MyDrive/hallway_lighting_datasets/MID_dataset/multi_illumination_test_mip2_jpg.zip')

DATASET_INPUTS = {
    'nyu_depth_v2': '',
    'mid_intrinsics': '',
}

IMAGE_SIZE = (256, 256)
BATCH_SIZE = 4
NUM_WORKERS = 2
TRAIN_EPOCHS = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
USE_PRETRAINED_BACKBONE = True
USE_COORD_CHANNELS = True
FIXTURE_COUNT = 3
FLOOR_Y = 0.72
POINT_START_X = 0.20
POINT_END_X = 0.80
MAX_VIS_EXAMPLES = 4
GRAD_CLIP_NORM = 1.0
AMP_ENABLED = True
SEED = 42

# Pseudo-lux calibration for public-datasets-only supervision.
PSEUDO_LUX_MIN = 50.0
PSEUDO_LUX_MAX = 300.0

# Photometric carbon prototype parameters.
HALLWAY_FLOOR_AREA_M2 = 12.0
LUMINOUS_EFFICACY_LM_PER_W = 110.0
UTILIZATION_FACTOR = 0.60
MAINTENANCE_FACTOR = 0.80
GRID_EMISSION_FACTOR_KG_PER_KWH = 0.40
PHOTO_INTERVAL_MINUTES = 5
INTERVAL_HOURS = PHOTO_INTERVAL_MINUTES / 60.0
CARBON_FACTOR_KG_PER_KWH = GRID_EMISSION_FACTOR_KG_PER_KWH
FIXTURE_COUNT_FOR_DEPLOYMENT = 3
DEFER_SECONDS_ON_OCCLUSION = 20
MAX_DEFER_SECONDS = 180
MAX_FLOOR_OCCLUSION_RATIO = 0.08

RESUME_CHECKPOINT_PATH = ''
SINGLE_IMAGE_PATH = ''
ONNX_EXPORT_PATH = str(EXPORT_DIR / 'hallway_multitask_unet_drive_prototype.onnx')
BEST_MODEL_PATH = CHECKPOINT_DIR / 'best_model.pt'
LAST_MODEL_PATH = CHECKPOINT_DIR / 'last_model.pt'
HISTORY_PATH = RUN_ROOT / 'training_history.json'

RUN_TRAINING = True
RUN_VALIDATION = True
RUN_TEST = True
RUN_SINGLE_IMAGE_INFERENCE = False
RUN_ONNX_EXPORT = False

if not IN_COLAB:
    ensure_run_directories()
print(f'RUN_ROOT: {RUN_ROOT}')
print(f'CACHE_ROOT: {CACHE_ROOT}')


## 3. Google Drive Mount

This notebook expects the NYU and MID zip files to be present in Google Drive at the user-provided paths.


In [ ]:
if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)
    ensure_run_directories()
    print(f'Google Drive mounted. Saving runs to: {RUN_ROOT}')
else:
    ensure_run_directories()
    print(f'google.colab is not available in this environment. Saving runs to: {RUN_ROOT}')


## 4. Extract NYU And MID From Google Drive

This cell unzips both datasets inside the runtime and previews the extracted roots.


In [ ]:
from pathlib import Path
import zipfile


def safe_unzip(zip_path: str | Path, destination: str | Path) -> Path:
    zip_path = Path(zip_path)
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as handle:
        handle.extractall(destination)
    return destination


def list_relative_paths(root: Path, limit: int = 25) -> list[str]:
    if not root.exists():
        return []
    items = sorted(root.rglob('*'))
    return [str(path.relative_to(root)) for path in items[:limit]]


if not NYU_DRIVE_ZIP.exists():
    raise FileNotFoundError(f'NYU zip not found: {NYU_DRIVE_ZIP}')
if not MID_TRAIN_DRIVE_ZIP.exists():
    raise FileNotFoundError(f'MID train zip not found: {MID_TRAIN_DRIVE_ZIP}')
if not MID_TEST_DRIVE_ZIP.exists():
    raise FileNotFoundError(f'MID test zip not found: {MID_TEST_DRIVE_ZIP}')

NYU_EXTRACT_ROOT = PREPARED_ROOT / 'nyu_depth_v2' / 'archive'
MID_EXTRACT_ROOT = PREPARED_ROOT / 'mid_intrinsics'
MID_TRAIN_ROOT = MID_EXTRACT_ROOT / 'multi_illumination_train_mip2_jpg'
MID_TEST_ROOT = MID_EXTRACT_ROOT / 'multi_illumination_test_mip2_jpg'

safe_unzip(NYU_DRIVE_ZIP, NYU_EXTRACT_ROOT)
safe_unzip(MID_TRAIN_DRIVE_ZIP, MID_TRAIN_ROOT)
safe_unzip(MID_TEST_DRIVE_ZIP, MID_TEST_ROOT)

DATASET_INPUTS['nyu_depth_v2'] = str(NYU_EXTRACT_ROOT)
DATASET_INPUTS['mid_intrinsics'] = str(MID_EXTRACT_ROOT)

extraction_rows = [
    {
        'dataset_name': 'nyu_depth_v2',
        'source_zip': str(NYU_DRIVE_ZIP),
        'extracted_root': str(NYU_EXTRACT_ROOT),
        'sample_files': list_relative_paths(NYU_EXTRACT_ROOT),
    },
    {
        'dataset_name': 'mid_intrinsics',
        'source_zip': f'{MID_TRAIN_DRIVE_ZIP} | {MID_TEST_DRIVE_ZIP}',
        'extracted_root': str(MID_EXTRACT_ROOT),
        'sample_files': list_relative_paths(MID_EXTRACT_ROOT),
    },
]

display(pd.DataFrame(extraction_rows))


## 5. Utilities And Manifest Builders

This section defines the notebook-local dataset utilities, archive helpers, geometry utilities, metrics, checkpointing, and manifest builders.


In [ ]:
IMAGE_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.exr')
ARRAY_EXTENSIONS = ('.npy', '.npz')
JSON_EXTENSIONS = ('.json',)
TEXT_EXTENSIONS = ('.txt', '.csv')
MANIFEST_COLUMNS = [
    'sample_id', 'dataset_name', 'split', 'image_path', 'depth_path', 'floor_mask_path',
    'lux_map_path', 'avg_lux', 'low_lux_p5', 'high_lux_p95', 'point_targets_json',
    'albedo_path', 'reflectance_path', 'shading_path', 'gloss_path', 'measured_power_w',
    'interval_hours', 'notes'
]


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_torch_generator(seed: int) -> torch.Generator:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_json(payload: dict[str, Any] | list[Any], path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        json.dump(payload, handle, indent=2)
    return path


def save_training_history(history: dict[str, Any], path: str | Path) -> Path:
    return save_json(history, path)


def save_checkpoint(
    model: nn.Module,
    optimizer: torch.optim.Optimizer | None,
    epoch: int,
    path: str | Path,
    scheduler: torch.optim.lr_scheduler.LRScheduler | None = None,
    scaler: torch.cuda.amp.GradScaler | None = None,
    history: dict[str, Any] | None = None,
    extra_state: dict[str, Any] | None = None,
) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': None if optimizer is None else optimizer.state_dict(),
            'scheduler_state_dict': None if scheduler is None else scheduler.state_dict(),
            'scaler_state_dict': None if scaler is None else scaler.state_dict(),
            'history': history or {},
            'extra_state': extra_state or {},
        },
        path,
    )
    return path


def load_checkpoint(
    path: str | Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
    scheduler: torch.optim.lr_scheduler.LRScheduler | None = None,
    scaler: torch.cuda.amp.GradScaler | None = None,
    map_location: str | torch.device = 'cpu',
) -> dict[str, Any]:
    checkpoint = torch.load(Path(path), map_location=map_location)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer is not None and checkpoint.get('optimizer_state_dict') is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if scheduler is not None and checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    if scaler is not None and checkpoint.get('scaler_state_dict') is not None:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
    return checkpoint


@dataclass(frozen=True)
class PreparedDatasetInput:
    dataset_name: str
    source_path: Path
    input_type: str
    prepared_root: Path
    primary_file: Path | None = None


def detect_input_type(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Dataset input does not exist: {path}')
    if path.is_dir():
        return 'directory'
    suffixes = [suffix.lower() for suffix in path.suffixes]
    if suffixes[-2:] == ['.tar', '.gz']:
        return 'tar.gz'
    if path.suffix.lower() == '.tgz':
        return 'tgz'
    if path.suffix.lower() == '.tar':
        return 'tar'
    if path.suffix.lower() == '.zip':
        return 'zip'
    if path.suffix.lower() == '.mat':
        return 'mat'
    if path.suffix.lower() == '.csv':
        return 'csv'
    raise ValueError(f'Unsupported dataset input type: {path}')


def _normalized_destination_name(path: Path) -> str:
    name = path.name
    for suffix in ('.tar.gz', '.tgz', '.zip', '.tar', '.mat', '.csv'):
        if name.endswith(suffix):
            return name[:-len(suffix)]
    return path.stem


def _ensure_within_directory(path: Path, directory: Path) -> None:
    resolved_path = path.resolve()
    resolved_directory = directory.resolve()
    if resolved_directory not in resolved_path.parents and resolved_path != resolved_directory:
        raise ValueError(f'Unsafe archive member path detected: {resolved_path}')


def _safe_extract_zip(archive_path: Path, destination_dir: Path) -> None:
    with zipfile.ZipFile(archive_path, 'r') as handle:
        for member in handle.infolist():
            _ensure_within_directory(destination_dir / member.filename, destination_dir)
        handle.extractall(destination_dir)


def _safe_extract_tar(archive_path: Path, destination_dir: Path) -> None:
    with tarfile.open(archive_path, 'r:*') as handle:
        for member in handle.getmembers():
            _ensure_within_directory(destination_dir / member.name, destination_dir)
        handle.extractall(destination_dir)


def extract_archive(archive_path: str | Path, destination_dir: str | Path, overwrite: bool = False) -> Path:
    archive_path = Path(archive_path)
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents=True, exist_ok=True)
    if any(destination_dir.iterdir()) and not overwrite:
        return destination_dir
    input_type = detect_input_type(archive_path)
    if input_type == 'zip':
        _safe_extract_zip(archive_path, destination_dir)
    elif input_type in {'tar', 'tar.gz', 'tgz'}:
        _safe_extract_tar(archive_path, destination_dir)
    else:
        raise ValueError(f'Cannot extract unsupported archive type: {input_type}')
    return destination_dir


def stage_mat_file(mat_path: str | Path, destination_dir: str | Path, overwrite: bool = False) -> Path:
    mat_path = Path(mat_path)
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents=True, exist_ok=True)
    destination_file = destination_dir / mat_path.name
    if destination_file.exists() and not overwrite:
        return destination_dir
    shutil.copy2(mat_path, destination_file)
    return destination_dir


def prepare_dataset_input(
    input_path: str | Path,
    dataset_name: str,
    working_dir: str | Path,
    overwrite: bool = False,
) -> PreparedDatasetInput:
    input_path = Path(input_path).expanduser().resolve()
    working_dir = Path(working_dir).expanduser().resolve()
    working_dir.mkdir(parents=True, exist_ok=True)
    input_type = detect_input_type(input_path)

    if input_type == 'directory':
        return PreparedDatasetInput(dataset_name, input_path, input_type, input_path, None)
    if input_type == 'csv':
        return PreparedDatasetInput(dataset_name, input_path, input_type, input_path.parent, input_path)

    destination_dir = working_dir / dataset_name / _normalized_destination_name(input_path)
    if input_type == 'mat':
        stage_mat_file(input_path, destination_dir, overwrite=overwrite)
        return PreparedDatasetInput(dataset_name, input_path, input_type, destination_dir, destination_dir / input_path.name)

    extract_archive(input_path, destination_dir, overwrite=overwrite)
    return PreparedDatasetInput(dataset_name, input_path, input_type, destination_dir, None)


def _normalize_path_value(value: Any) -> str:
    if value is None:
        return ''
    if isinstance(value, float) and np.isnan(value):
        return ''
    text = str(value).strip()
    if not text:
        return ''
    return str(Path(text))


def _normalize_float_value(value: Any) -> float | None:
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    if isinstance(value, str) and not value.strip():
        return None
    return float(value)


def make_manifest_row(dataset_name: str, sample_id: str, image_path: str | Path, split: str = 'unspecified', **kwargs: Any) -> dict[str, Any]:
    row = {column: '' for column in MANIFEST_COLUMNS}
    for numeric_key in ('avg_lux', 'low_lux_p5', 'high_lux_p95', 'measured_power_w', 'interval_hours'):
        row[numeric_key] = None
    row['dataset_name'] = dataset_name
    row['sample_id'] = sample_id
    row['split'] = split
    row['image_path'] = str(Path(image_path).resolve())
    for key, value in kwargs.items():
        if key not in row:
            continue
        if key in ('avg_lux', 'low_lux_p5', 'high_lux_p95', 'measured_power_w', 'interval_hours'):
            row[key] = _normalize_float_value(value)
        elif key.endswith('_path') or key.endswith('_json'):
            row[key] = '' if not value else str(Path(value).resolve())
        else:
            row[key] = '' if value is None else str(value)
    return row


def create_manifest_dataframe(rows: Sequence[dict[str, Any]]) -> pd.DataFrame:
    manifest = pd.DataFrame(rows)
    if manifest.empty:
        return pd.DataFrame(columns=MANIFEST_COLUMNS)
    for column in MANIFEST_COLUMNS:
        if column not in manifest.columns:
            manifest[column] = None if column in ('avg_lux', 'low_lux_p5', 'high_lux_p95', 'measured_power_w', 'interval_hours') else ''
    return manifest[MANIFEST_COLUMNS]


def save_manifest(manifest: pd.DataFrame, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(path, index=False)
    return path


def infer_split_from_path(path: str | Path) -> str:
    path = Path(path)
    for part in path.parts:
        lower = part.lower()
        if lower == 'train':
            return 'train'
        if lower in {'val', 'valid', 'validation'}:
            return 'val'
        if lower == 'test':
            return 'test'
    return 'unspecified'


def iter_files(directory: str | Path, extensions: tuple[str, ...], recursive: bool = False) -> list[Path]:
    directory = Path(directory)
    if not directory.exists():
        return []
    iterator = directory.rglob('*') if recursive else directory.iterdir()
    return sorted(path for path in iterator if path.is_file() and path.suffix.lower() in extensions)


def iter_scene_directories(dataset_root: str | Path) -> list[Path]:
    dataset_root = Path(dataset_root)
    scene_dirs = []
    for candidate in [dataset_root, *sorted(path for path in dataset_root.rglob('*') if path.is_dir())]:
        file_count = sum(1 for _ in candidate.iterdir()) if candidate.exists() else 0
        if file_count > 0:
            scene_dirs.append(candidate)
    return scene_dirs


def find_first_file(directory: str | Path, keywords: Sequence[str], extensions: tuple[str, ...] = IMAGE_EXTENSIONS, recursive: bool = False) -> Path | None:
    candidates = iter_files(directory, extensions=extensions, recursive=recursive)
    keywords = tuple(keyword.lower() for keyword in keywords)
    for path in candidates:
        lower_name = path.name.lower()
        if all(keyword in lower_name for keyword in keywords[:1]) or any(keyword in lower_name for keyword in keywords):
            return path
    return None


def load_text_split_assignments(root: str | Path) -> dict[str, str]:
    root = Path(root)
    assignments: dict[str, str] = {}
    split_files = {
        'train': ('train.txt',),
        'val': ('val.txt', 'valid.txt', 'validation.txt'),
        'test': ('test.txt',),
    }
    for split_name, file_names in split_files.items():
        for file_name in file_names:
            for candidate in root.rglob(file_name):
                for line in candidate.read_text(encoding='utf-8').splitlines():
                    item = line.strip()
                    if not item or item.startswith('#'):
                        continue
                    assignments[Path(item).stem] = split_name
    return assignments


def _find_subdirectory(root: Path, names: tuple[str, ...]) -> Path | None:
    names = {name.lower() for name in names}
    for candidate in [root, *sorted(path for path in root.rglob('*') if path.is_dir())]:
        if candidate.name.lower() in names:
            return candidate
    return None


def _find_nyu_mat_file(dataset_root: Path) -> Path | None:
    candidates = sorted(dataset_root.rglob('*.mat'))
    if not candidates:
        return None
    preferred = [candidate for candidate in candidates if 'nyu' in candidate.name.lower()]
    return preferred[0] if preferred else candidates[0]


def _find_nyu_archive_file(dataset_root: Path) -> Path | None:
    archive_candidates: list[Path] = []
    for pattern in ('*.zip', '*.tar', '*.tgz', '*.tar.gz'):
        archive_candidates.extend(sorted(dataset_root.rglob(pattern)))
    if not archive_candidates:
        return None
    preferred = [candidate for candidate in archive_candidates if 'nyu' in candidate.name.lower() or 'depth' in candidate.name.lower()]
    return preferred[0] if preferred else archive_candidates[0]


def resolve_nyu_dataset_root(dataset_input: str | Path, working_dir: str | Path, overwrite: bool = False) -> Path:
    dataset_input = Path(dataset_input).expanduser().resolve()
    working_dir = Path(working_dir).expanduser().resolve()

    if dataset_input.is_file():
        prepared = prepare_dataset_input(dataset_input, 'nyu_depth_v2', working_dir, overwrite=overwrite)
        return prepared.prepared_root

    rgb_dir = _find_subdirectory(dataset_input, ('rgb', 'images'))
    depth_dir = _find_subdirectory(dataset_input, ('depth', 'depths', 'rawdepths'))
    if rgb_dir is not None and depth_dir is not None:
        return dataset_input

    mat_path = _find_nyu_mat_file(dataset_input)
    if mat_path is not None:
        return _extract_nyu_mat(working_dir / 'nyu_depth_v2', mat_path, overwrite=overwrite)

    archive_path = _find_nyu_archive_file(dataset_input)
    if archive_path is not None:
        prepared = prepare_dataset_input(archive_path, 'nyu_depth_v2', working_dir, overwrite=overwrite)
        prepared_root = prepared.prepared_root
        rgb_dir = _find_subdirectory(prepared_root, ('rgb', 'images'))
        depth_dir = _find_subdirectory(prepared_root, ('depth', 'depths', 'rawdepths'))
        if rgb_dir is not None and depth_dir is not None:
            return prepared_root
        mat_path = _find_nyu_mat_file(prepared_root)
        if mat_path is not None:
            return _extract_nyu_mat(working_dir / 'nyu_depth_v2', mat_path, overwrite=overwrite)

    image_only_candidate: Path | None = None
    for candidate in [dataset_input, *sorted(path for path in dataset_input.rglob('*') if path.is_dir())]:
        rgb_dir = _find_subdirectory(candidate, ('rgb', 'images'))
        depth_dir = _find_subdirectory(candidate, ('depth', 'depths', 'rawdepths'))
        if rgb_dir is not None and depth_dir is not None:
            return candidate
        mat_path = _find_nyu_mat_file(candidate)
        if mat_path is not None:
            return _extract_nyu_mat(working_dir / 'nyu_depth_v2', mat_path, overwrite=overwrite)
        rgb_files, depth_by_stem = discover_generic_nyu_pairs(candidate)
        if rgb_files and depth_by_stem:
            return candidate
        if image_only_candidate is None and discover_nyu_rgb_only_files(candidate):
            image_only_candidate = candidate

    if image_only_candidate is not None:
        print(f'Falling back to image-only NYU directory: {image_only_candidate}')
        return image_only_candidate

    raise FileNotFoundError(
        f'Could not locate NYU rgb/depth content, .mat file, extractable archive, or generic RGB/depth pairs under {dataset_input}. '
        'Point DATASET_INPUTS["nyu_depth_v2"] to the actual NYU dataset folder, .mat file, or archive.'
    )


def _normalize_nyu_pair_key(path: Path) -> str:
    key = path.stem.lower()
    for token in ('_depth', '-depth', ' depth', '_rgb', '-rgb', ' rgb', '_color', '-color', ' color', '_image', '-image', ' image', '_img', '-img', ' img', '_rawdepth', '-rawdepth', ' rawdepth'):
        key = key.replace(token, '')
    return key.strip('_- ')


def _looks_like_depth_path(path: Path) -> bool:
    text = str(path).lower()
    return any(token in text for token in ('depth', 'rawdepth', 'depths', 'dmap')) or path.suffix.lower() in ARRAY_EXTENSIONS


def discover_generic_nyu_pairs(dataset_root: Path) -> tuple[list[Path], dict[str, Path]]:
    image_files = sorted(path for path in dataset_root.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
    array_files = sorted(path for path in dataset_root.rglob('*') if path.is_file() and path.suffix.lower() in ARRAY_EXTENSIONS)

    depth_candidates = [path for path in image_files if _looks_like_depth_path(path)] + array_files
    rgb_candidates = [path for path in image_files if not _looks_like_depth_path(path)]

    depth_by_key: dict[str, Path] = {}
    for path in depth_candidates:
        key = _normalize_nyu_pair_key(path)
        depth_by_key.setdefault(key, path)

    rgb_files: list[Path] = []
    paired_depths: dict[str, Path] = {}
    for path in rgb_candidates:
        key = _normalize_nyu_pair_key(path)
        depth_path = depth_by_key.get(key)
        if depth_path is None:
            continue
        rgb_files.append(path)
        paired_depths[path.stem] = depth_path

    return rgb_files, paired_depths


def discover_nyu_rgb_only_files(dataset_root: Path) -> list[Path]:
    image_files = sorted(path for path in dataset_root.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
    rgb_candidates = [path for path in image_files if not _looks_like_depth_path(path)]
    if rgb_candidates:
        return rgb_candidates
    return image_files


def _image_layout(shape: tuple[int, ...]) -> str:
    if len(shape) != 4:
        raise ValueError(f'Expected 4D NYU image tensor, got {shape}')
    if shape[2] == 3 and shape[-1] > 3:
        return 'hwcn'
    if shape[-1] == 3:
        return 'nhwc'
    if shape[1] == 3:
        return 'nchw'
    if shape[0] == 3:
        return 'chwn'
    raise ValueError(f'Could not infer NYU image layout from {shape}')


def _depth_layout(shape: tuple[int, ...], sample_count: int) -> str:
    if len(shape) != 3:
        raise ValueError(f'Expected 3D NYU depth tensor, got {shape}')
    if shape[0] == sample_count:
        return 'nhw'
    if shape[-1] == sample_count:
        return 'hwn'
    raise ValueError(f'Could not infer NYU depth layout from {shape}')


def _extract_image_sample(images: Any, index: int, layout: str) -> np.ndarray:
    if layout == 'hwcn':
        sample = np.asarray(images[..., index])
    elif layout == 'nhwc':
        sample = np.asarray(images[index, ...])
    elif layout == 'nchw':
        sample = np.asarray(images[index, ...]).transpose(1, 2, 0)
    elif layout == 'chwn':
        sample = np.asarray(images[:, :, :, index]).transpose(1, 2, 0)
    else:
        raise ValueError(layout)
    return np.asarray(np.clip(sample, 0, 255), dtype=np.uint8)


def _extract_depth_sample(depths: Any, index: int, layout: str) -> np.ndarray:
    if layout == 'nhw':
        sample = np.asarray(depths[index, ...])
    elif layout == 'hwn':
        sample = np.asarray(depths[..., index])
    else:
        raise ValueError(layout)
    return np.asarray(sample, dtype=np.float32)


def _extract_nyu_mat(dataset_root: Path, mat_path: Path, overwrite: bool = False) -> Path:
    output_root = dataset_root / 'nyu_depth_v2_extracted'
    rgb_dir = output_root / 'rgb'
    depth_dir = output_root / 'depth'
    if rgb_dir.exists() and depth_dir.exists() and any(rgb_dir.iterdir()) and any(depth_dir.iterdir()) and not overwrite:
        return output_root
    output_root.mkdir(parents=True, exist_ok=True)
    rgb_dir.mkdir(parents=True, exist_ok=True)
    depth_dir.mkdir(parents=True, exist_ok=True)
    try:
        payload = loadmat(mat_path)
        images = payload.get('images')
        depths = payload.get('rawDepths') if payload.get('rawDepths') is not None else payload.get('depths')
        if images is None or depths is None:
            raise ValueError('NYU .mat file must contain images and depths/rawDepths')
        image_layout = _image_layout(tuple(images.shape))
        sample_count = images.shape[-1] if image_layout in {'hwcn', 'chwn'} else images.shape[0]
        depth_layout = _depth_layout(tuple(depths.shape), sample_count)
        for index in range(sample_count):
            sample_id = f'{index:06d}'
            Image.fromarray(_extract_image_sample(images, index, image_layout)).save(rgb_dir / f'{sample_id}.png')
            np.save(depth_dir / f'{sample_id}.npy', _extract_depth_sample(depths, index, depth_layout))
        return output_root
    except NotImplementedError:
        import h5py
        with h5py.File(mat_path, 'r') as handle:
            images = handle['images']
            depths = handle['rawDepths'] if 'rawDepths' in handle else handle['depths']
            image_layout = _image_layout(tuple(images.shape))
            sample_count = images.shape[-1] if image_layout in {'hwcn', 'chwn'} else images.shape[0]
            depth_layout = _depth_layout(tuple(depths.shape), sample_count)
            for index in range(sample_count):
                sample_id = f'{index:06d}'
                Image.fromarray(_extract_image_sample(images, index, image_layout)).save(rgb_dir / f'{sample_id}.png')
                np.save(depth_dir / f'{sample_id}.npy', _extract_depth_sample(depths, index, depth_layout))
        return output_root


def build_nyu_depth_v2_manifest(dataset_root: str | Path, output_path: str | Path | None = None) -> pd.DataFrame:
    dataset_root = Path(dataset_root)
    resolved_root = resolve_nyu_dataset_root(dataset_root, PREPARED_ROOT, overwrite=False)
    rgb_dir = _find_subdirectory(resolved_root, ('rgb', 'images'))
    depth_dir = _find_subdirectory(resolved_root, ('depth', 'depths', 'rawdepths'))

    if rgb_dir is not None and depth_dir is not None:
        rgb_files = sorted(path for path in rgb_dir.rglob('*') if path.suffix.lower() in IMAGE_EXTENSIONS)
        depth_files = sorted(path for path in depth_dir.rglob('*') if path.suffix.lower() in IMAGE_EXTENSIONS + ARRAY_EXTENSIONS)
        depth_by_stem = {path.stem: path for path in depth_files}
    else:
        rgb_files, depth_by_stem = discover_generic_nyu_pairs(resolved_root)

    split_assignments = load_text_split_assignments(dataset_root)
    rows = []
    paired_count = 0
    image_only_count = 0
    for image_path in rgb_files:
        depth_path = depth_by_stem.get(image_path.stem)
        if depth_path is not None:
            paired_count += 1
            rows.append(
                make_manifest_row(
                    dataset_name='nyu_depth_v2',
                    sample_id=image_path.stem,
                    image_path=image_path,
                    split=split_assignments.get(image_path.stem, 'unspecified'),
                    depth_path=depth_path,
                    notes='Official NYU Depth V2 RGB-D sample.',
                )
            )

    if not rows:
        rgb_only_files = discover_nyu_rgb_only_files(resolved_root)
        for image_path in rgb_only_files:
            image_only_count += 1
            rows.append(
                make_manifest_row(
                    dataset_name='nyu_depth_v2',
                    sample_id=image_path.stem,
                    image_path=image_path,
                    split=split_assignments.get(image_path.stem, 'unspecified'),
                    depth_path=None,
                    notes='NYU image-only fallback sample. Depth pairing was not available in the extracted archive layout.',
                )
            )

    if not rows:
        print(f'No NYU RGB files found under: {resolved_root}')
        for entry in list(sorted(resolved_root.rglob('*')))[:120]:
            print(entry)
        raise ValueError(f'NYU Depth V2 manifest generation found no usable image samples under {resolved_root}.')

    if image_only_count > 0:
        print(f'Built NYU manifest with {image_only_count} image-only samples from {resolved_root}.')
    elif paired_count > 0:
        print(f'Built NYU manifest with {paired_count} RGB-depth pairs from {resolved_root}.')

    manifest = create_manifest_dataframe(rows)
    if output_path is not None:
        save_manifest(manifest, output_path)
    return manifest


def build_mid_intrinsics_manifest(dataset_root: str | Path, output_path: str | Path | None = None) -> pd.DataFrame:
    dataset_root = Path(dataset_root)
    rows = []
    seen = set()
    for scene_dir in iter_scene_directories(dataset_root):
        image_path = find_first_file(scene_dir, keywords=('input', 'rgb', 'image', 'img', 'jpg', 'jpeg', 'ldr', 'tone'), recursive=False)
        if image_path is None:
            image_candidates = iter_files(scene_dir, IMAGE_EXTENSIONS, recursive=False)
            image_path = image_candidates[0] if image_candidates else None
        albedo_path = find_first_file(scene_dir, keywords=('albedo', 'reflectance'), recursive=False)
        shading_path = find_first_file(scene_dir, keywords=('shading',), recursive=False)
        gloss_path = find_first_file(scene_dir, keywords=('gloss', 'specular', 'roughness'), recursive=False)
        if image_path is None or image_path in seen:
            continue
        seen.add(image_path)
        sample_id = scene_dir.relative_to(dataset_root).as_posix().replace('/', '_') or image_path.stem
        note = 'MID / MIDIntrinsics sample.'
        if albedo_path is None and shading_path is None and gloss_path is None:
            note = 'MID sample without explicit intrinsic targets.'
        rows.append(
            make_manifest_row(
                dataset_name='mid_intrinsics',
                sample_id=sample_id,
                image_path=image_path,
                split=infer_split_from_path(image_path),
                albedo_path=albedo_path if albedo_path else '',
                reflectance_path=albedo_path if albedo_path else '',
                shading_path=shading_path if shading_path else '',
                gloss_path=gloss_path if gloss_path else '',
                notes=note,
            )
        )
    if not rows:
        raise ValueError(f'No MID/MIDIntrinsics samples found under {dataset_root}')
    manifest = create_manifest_dataframe(rows)
    if output_path is not None:
        save_manifest(manifest, output_path)
    return manifest



DATASET_BUILDERS = {
    'nyu_depth_v2': build_nyu_depth_v2_manifest,
    'mid_intrinsics': build_mid_intrinsics_manifest,
}


def build_all_dataset_manifests(dataset_inputs: Mapping[str, str | Path], working_dir: str | Path, output_dir: str | Path) -> dict[str, pd.DataFrame]:
    results = {}
    working_dir = Path(working_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for dataset_name, raw_input in dataset_inputs.items():
        if raw_input in ('', None):
            continue
        prepared = prepare_dataset_input(raw_input, dataset_name, working_dir, overwrite=False)
        builder_input = prepared.primary_file if prepared.input_type == 'csv' else prepared.prepared_root
        manifest = DATASET_BUILDERS[dataset_name](builder_input, output_dir / f'{dataset_name}.csv')
        results[dataset_name] = manifest
    return results


def denormalize_image(image: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor([0.485, 0.456, 0.406], device=image.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=image.device).view(3, 1, 1)
    return image * std + mean


def prepare_display_image(image: torch.Tensor | np.ndarray, denormalize: bool = True) -> np.ndarray:
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu()
        if denormalize and image.ndim == 3 and image.shape[0] == 3:
            image = denormalize_image(image).clamp(0.0, 1.0)
        image = image.numpy()
    image = np.asarray(image)
    if image.ndim == 3 and image.shape[0] in {1, 3}:
        image = np.transpose(image, (1, 2, 0))
    if image.ndim == 3 and image.shape[-1] == 1:
        image = image[..., 0]
    if image.ndim == 3:
        return np.clip(image, 0.0, 1.0)
    return image


def show_image(image: torch.Tensor | np.ndarray, title: str = 'Image') -> None:
    image_np = prepare_display_image(image)
    plt.figure(figsize=(6, 4))
    plt.imshow(image_np, cmap='gray' if image_np.ndim == 2 else None)
    plt.title(title)
    plt.axis('off')
    plt.show()


def save_figure(fig: plt.Figure, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    return path


@dataclass(frozen=True)
class PointTarget:
    name: str
    x: float
    y: float
    group: str


def build_coordinate_channels(height: int, width: int, device: torch.device | None = None, dtype: torch.dtype | None = None) -> torch.Tensor:
    y_values = torch.linspace(-1.0, 1.0, steps=height, device=device, dtype=dtype)
    x_values = torch.linspace(-1.0, 1.0, steps=width, device=device, dtype=dtype)
    y_grid, x_grid = torch.meshgrid(y_values, x_values, indexing='ij')
    return torch.stack([x_grid, y_grid], dim=0)


def expand_coordinate_channels(batch_size: int, height: int, width: int, device: torch.device | None = None, dtype: torch.dtype | None = None) -> torch.Tensor:
    return build_coordinate_channels(height, width, device=device, dtype=dtype).unsqueeze(0).repeat(batch_size, 1, 1, 1)


def canonical_fixture_positions(fixture_count: int, start_x: float, end_x: float, floor_y: float) -> list[tuple[float, float]]:
    if fixture_count == 1:
        return [((start_x + end_x) / 2.0, floor_y)]
    spacing = (end_x - start_x) / float(fixture_count - 1)
    return [(start_x + spacing * index, floor_y) for index in range(fixture_count)]


def canonical_between_fixture_positions(fixture_count: int, start_x: float, end_x: float, floor_y: float) -> list[tuple[float, float]]:
    fixture_positions = canonical_fixture_positions(fixture_count, start_x, end_x, floor_y)
    return [((left[0] + right[0]) / 2.0, floor_y) for left, right in zip(fixture_positions[:-1], fixture_positions[1:], strict=False)]


def build_canonical_point_targets(fixture_count: int = 3, floor_y: float = 0.72, start_x: float = 0.2, end_x: float = 0.8) -> list[PointTarget]:
    points = []
    for index, (x_value, y_value) in enumerate(canonical_fixture_positions(fixture_count, start_x, end_x, floor_y), start=1):
        points.append(PointTarget(name=f'under_fixture_{index}', x=x_value, y=y_value, group='under_fixture'))
    for index, (x_value, y_value) in enumerate(canonical_between_fixture_positions(fixture_count, start_x, end_x, floor_y), start=1):
        points.append(PointTarget(name=f'between_fixture_{index}_{index + 1}', x=x_value, y=y_value, group='between_fixture'))
    return points


def load_point_target_values(path: str | Path) -> dict[str, float]:
    with Path(path).open('r', encoding='utf-8') as handle:
        payload = json.load(handle)
    if not isinstance(payload, dict):
        raise ValueError('Point target JSON must be an object mapping names to values.')
    values = {}
    for key, value in payload.items():
        if isinstance(value, dict):
            if 'value' not in value:
                raise ValueError(f'Point target {key} object must contain a value field.')
            values[key] = float(value['value'])
        else:
            values[key] = float(value)
    return values


def _as_lux_tensor(lux_map: torch.Tensor | np.ndarray) -> torch.Tensor:
    tensor = torch.from_numpy(lux_map) if isinstance(lux_map, np.ndarray) else lux_map
    if tensor.ndim == 2:
        tensor = tensor.unsqueeze(0).unsqueeze(0)
    elif tensor.ndim == 3:
        if tensor.shape[0] == 1:
            tensor = tensor.unsqueeze(0)
        else:
            tensor = tensor.unsqueeze(1)
    elif tensor.ndim != 4:
        raise ValueError(f'Expected 2D, 3D, or 4D lux map, got {tuple(tensor.shape)}')
    if tensor.shape[1] != 1:
        raise ValueError(f'Expected single-channel lux map, got {tuple(tensor.shape)}')
    return tensor.float()


def sample_point_values_batch(lux_map: torch.Tensor, points: Sequence[PointTarget]) -> dict[str, torch.Tensor]:
    lux_tensor = _as_lux_tensor(lux_map)
    if not points:
        return {}
    batch_size = lux_tensor.shape[0]
    grid_coordinates = [(point.x * 2.0 - 1.0, point.y * 2.0 - 1.0) for point in points]
    grid = torch.tensor(grid_coordinates, device=lux_tensor.device, dtype=lux_tensor.dtype).view(1, len(points), 1, 2)
    grid = grid.repeat(batch_size, 1, 1, 1)
    sampled = F.grid_sample(lux_tensor, grid, mode='bilinear', padding_mode='border', align_corners=True)
    sampled = sampled.squeeze(1).squeeze(-1)
    return {point.name: sampled[:, index] for index, point in enumerate(points)}


def sample_values_at_points(lux_map: torch.Tensor | np.ndarray, points: Sequence[PointTarget]) -> dict[str, float]:
    lux_tensor = _as_lux_tensor(lux_map)
    if lux_tensor.shape[0] != 1:
        raise ValueError('sample_values_at_points expects a single sample.')
    values = sample_point_values_batch(lux_tensor, points)
    return {name: float(tensor[0].item()) for name, tensor in values.items()}


def summarize_lux_map(lux_map: torch.Tensor | np.ndarray, floor_mask: torch.Tensor | np.ndarray | None = None) -> dict[str, float]:
    lux = lux_map.detach().cpu().numpy() if isinstance(lux_map, torch.Tensor) else np.asarray(lux_map)
    if floor_mask is not None:
        mask = floor_mask.detach().cpu().numpy() if isinstance(floor_mask, torch.Tensor) else np.asarray(floor_mask)
        mask = mask.astype(bool)
        if mask.shape != lux.shape:
            mask = np.broadcast_to(mask, lux.shape)
        values = lux[mask]
    else:
        values = lux.reshape(-1)
    if values.size == 0:
        return {'avg_lux': 0.0, 'low_lux_p5': 0.0, 'high_lux_p95': 0.0, 'lux_discrepancy': 0.0}
    low = float(np.percentile(values, 5))
    high = float(np.percentile(values, 95))
    return {
        'avg_lux': float(np.mean(values)),
        'low_lux_p5': low,
        'high_lux_p95': high,
        'lux_discrepancy': high - low,
    }


def estimate_power_from_lux(avg_lux: float, floor_area_m2: float, watts_per_lux_m2: float) -> float:
    return avg_lux * floor_area_m2 * watts_per_lux_m2


def estimate_interval_energy_kwh(power_w: float, interval_hours: float) -> float:
    return (power_w / 1000.0) * interval_hours


def estimate_interval_carbon_kg(energy_kwh: float, carbon_factor_kg_per_kwh: float) -> float:
    return energy_kwh * carbon_factor_kg_per_kwh


## 6. Build NYU And MID Manifests

This cell builds manifests from the extracted Google Drive dataset folders and prints `head()` previews for both datasets.


In [ ]:
set_seed(SEED)


def unwrap_single_child_dirs(path: Path, max_depth: int = 4) -> Path:
    current = path
    for _ in range(max_depth):
        if not current.exists() or not current.is_dir():
            break
        children = [child for child in current.iterdir() if child.is_dir()]
        files = [child for child in current.iterdir() if child.is_file()]
        if len(children) == 1 and not files:
            current = children[0]
            continue
        break
    return current


DATASET_INPUTS_FOR_BUILD = {
    'nyu_depth_v2': str(unwrap_single_child_dirs(Path(DATASET_INPUTS['nyu_depth_v2']))),
    'mid_intrinsics': str(unwrap_single_child_dirs(Path(DATASET_INPUTS['mid_intrinsics']))),
}

MANIFESTS = build_all_dataset_manifests(DATASET_INPUTS_FOR_BUILD, PREPARED_ROOT, MANIFEST_DIR)


def assign_fallback_splits(manifest: pd.DataFrame) -> pd.DataFrame:
    manifest = manifest.copy()
    split_series = manifest['split'].fillna('').astype(str).str.lower()
    explicit = split_series.isin(['train', 'val', 'test']).sum()
    if explicit > 0:
        return manifest
    row_count = len(manifest)
    if row_count == 0:
        return manifest
    split_values = np.array(['train'] * row_count, dtype=object)
    if row_count >= 3:
        train_end = max(1, int(0.8 * row_count))
        val_end = max(train_end + 1, int(0.9 * row_count))
        val_end = min(val_end, row_count)
        split_values[train_end:val_end] = 'val'
        split_values[val_end:] = 'test'
    elif row_count == 2:
        split_values[1] = 'val'
    manifest['split'] = split_values
    return manifest


for dataset_name, manifest in list(MANIFESTS.items()):
    manifest = assign_fallback_splits(manifest)
    MANIFESTS[dataset_name] = manifest
    save_manifest(manifest, MANIFEST_DIR / f'{dataset_name}.csv')

MANIFEST_PATHS = {dataset_name: MANIFEST_DIR / f'{dataset_name}.csv' for dataset_name in MANIFESTS.keys()}

summary_rows = []
for dataset_name, manifest in MANIFESTS.items():
    summary_rows.append(
        {
            'dataset_name': dataset_name,
            'rows': len(manifest),
            'train_rows': int((manifest['split'].fillna('').astype(str).str.lower() == 'train').sum()),
            'val_rows': int((manifest['split'].fillna('').astype(str).str.lower() == 'val').sum()),
            'test_rows': int((manifest['split'].fillna('').astype(str).str.lower() == 'test').sum()),
            'has_depth': int(manifest['depth_path'].fillna('').astype(str).str.len().gt(0).sum()),
            'has_albedo': int(manifest['albedo_path'].fillna('').astype(str).str.len().gt(0).sum()),
            'has_gloss': int(manifest['gloss_path'].fillna('').astype(str).str.len().gt(0).sum()),
        }
    )

display(pd.DataFrame(summary_rows)) if summary_rows else print('No manifests were built.')
for dataset_name, manifest in MANIFESTS.items():
    print(f'Preview: {dataset_name}')
    display(manifest.head(5))


## 7. Dataset, Pseudo-Lux Supervision, Metrics, And Training Helpers

NYU provides geometry/depth priors. MID provides appearance priors. Because there are no direct hallway lux labels here, the notebook uses a pseudo-lux target to make the lux head trainable for a prototype.


In [ ]:
DATASET_SUPERVISION_RULES = {
    'nyu_depth_v2': {'depth', 'lux', 'stats', 'point'},
    'mid_intrinsics': {'albedo', 'gloss', 'lux', 'stats', 'point'},
}


def build_image_transform(image_size: tuple[int, int], normalize: bool = True):
    transform_steps = [
        transforms.Resize(image_size),
        transforms.ToTensor(),
    ]
    if normalize:
        transform_steps.append(transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]))
    return transforms.Compose(transform_steps)


def _read_rgb_image(path: str | Path) -> Image.Image:
    path = Path(path)
    try:
        return Image.open(path).convert('RGB')
    except Exception:
        array = np.asarray(imread(path))
        if array.ndim == 2:
            array = np.repeat(array[..., None], 3, axis=-1)
        if array.shape[-1] > 3:
            array = array[..., :3]
        if array.dtype != np.uint8:
            array = array.astype(np.float32)
            array = array - array.min()
            max_value = float(array.max())
            if max_value > 0:
                array = array / max_value
            array = (array * 255.0).clip(0, 255).astype(np.uint8)
        return Image.fromarray(array)


def _load_map_array(path: str | Path) -> np.ndarray:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == '.npy':
        return np.asarray(np.load(path))
    if suffix == '.npz':
        archive = np.load(path)
        if not archive.files:
            raise ValueError(f'Empty npz archive: {path}')
        return np.asarray(archive[archive.files[0]])
    return np.asarray(Image.open(path))


def _resize_tensor_map(array: np.ndarray, image_size: tuple[int, int], mode: str = 'bilinear', channels: int = 1) -> torch.Tensor:
    tensor = torch.as_tensor(array, dtype=torch.float32)
    if tensor.ndim == 2:
        tensor = tensor.unsqueeze(0).unsqueeze(0)
    elif tensor.ndim == 3:
        if tensor.shape[-1] in {1, 3}:
            tensor = tensor.permute(2, 0, 1).unsqueeze(0)
        else:
            tensor = tensor.unsqueeze(0)
    else:
        raise ValueError(f'Unsupported map shape {tuple(tensor.shape)}')
    tensor = F.interpolate(tensor, size=image_size, mode=mode, align_corners=False if mode != 'nearest' else None).squeeze(0)
    if channels == 1 and tensor.shape[0] != 1:
        tensor = tensor[:1]
    if channels == 3 and tensor.shape[0] == 1:
        tensor = tensor.repeat(3, 1, 1)
    return tensor


def _load_optional_dense_target(path_value: Any, image_size: tuple[int, int], channels: int, mode: str) -> torch.Tensor | None:
    if path_value is None:
        return None
    if isinstance(path_value, float) and np.isnan(path_value):
        return None
    path_text = str(path_value).strip()
    if not path_text:
        return None
    path = Path(path_text)
    if not path.exists():
        raise FileNotFoundError(f'Target file does not exist: {path}')
    return _resize_tensor_map(_load_map_array(path), image_size=image_size, mode=mode, channels=channels)


def _optional_scalar(value: Any) -> float | None:
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    if isinstance(value, str) and not value.strip():
        return None
    return float(value)


def _single_channel_map(tensor: torch.Tensor | None) -> torch.Tensor | None:
    if tensor is None:
        return None
    if tensor.ndim == 2:
        return tensor
    if tensor.ndim == 3 and tensor.shape[0] == 1:
        return tensor[0]
    if tensor.ndim == 3 and tensor.shape[0] >= 3:
        return 0.2126 * tensor[0] + 0.7152 * tensor[1] + 0.0722 * tensor[2]
    raise ValueError(f'Unsupported tensor shape for single-channel conversion: {tuple(tensor.shape)}')


def build_pseudo_lux_map(
    image: torch.Tensor,
    albedo: torch.Tensor | None = None,
    gloss: torch.Tensor | None = None,
    depth: torch.Tensor | None = None,
) -> torch.Tensor:
    image_rgb = denormalize_image(image).clamp(0.0, 1.0)
    luminance = 0.2126 * image_rgb[0] + 0.7152 * image_rgb[1] + 0.0722 * image_rgb[2]
    base = luminance

    albedo_gray = _single_channel_map(albedo)
    if albedo_gray is not None:
        albedo_gray = albedo_gray.clamp(0.0, 1.0)
        base = base / torch.clamp(albedo_gray + 0.2, min=0.2, max=1.2)

    gloss_gray = _single_channel_map(gloss)
    if gloss_gray is not None:
        gloss_gray = gloss_gray.clamp(0.0, 1.0)
        base = base * (1.0 + 0.15 * gloss_gray)

    depth_gray = _single_channel_map(depth)
    if depth_gray is not None:
        depth_gray = depth_gray.float()
        depth_gray = depth_gray - depth_gray.min()
        depth_gray = depth_gray / (depth_gray.max() + 1e-6)
        base = 0.85 * base + 0.15 * (1.0 - depth_gray)

    base = base - base.min()
    base = base / (base.max() + 1e-6)
    pseudo_lux = PSEUDO_LUX_MIN + (PSEUDO_LUX_MAX - PSEUDO_LUX_MIN) * base
    return pseudo_lux.unsqueeze(0)


class ManifestMultitaskDataset(Dataset):
    def __init__(self, manifest: pd.DataFrame, image_size: tuple[int, int] = (256, 256), normalize_images: bool = True) -> None:
        self.manifest = manifest.reset_index(drop=True).copy()
        self.image_size = image_size
        self.image_transform = build_image_transform(image_size=image_size, normalize=normalize_images)

    def __len__(self) -> int:
        return len(self.manifest)

    def __getitem__(self, index: int) -> dict[str, Any]:
        row = self.manifest.iloc[index]
        image_path = Path(row['image_path'])
        if not image_path.exists():
            raise FileNotFoundError(f'Image file does not exist: {image_path}')
        image = self.image_transform(_read_rgb_image(image_path))
        sample = {
            'sample_id': str(row['sample_id']),
            'dataset_name': str(row['dataset_name']),
            'split': str(row['split']),
            'image_path': str(image_path),
            'image': image,
            'depth': _load_optional_dense_target(row.get('depth_path'), self.image_size, channels=1, mode='bilinear'),
            'floor_mask': _load_optional_dense_target(row.get('floor_mask_path'), self.image_size, channels=1, mode='nearest'),
            'lux_map': _load_optional_dense_target(row.get('lux_map_path'), self.image_size, channels=1, mode='bilinear'),
            'albedo': _load_optional_dense_target(row.get('albedo_path') or row.get('reflectance_path'), self.image_size, channels=3, mode='bilinear'),
            'gloss': _load_optional_dense_target(row.get('gloss_path'), self.image_size, channels=1, mode='bilinear'),
            'avg_lux': _optional_scalar(row.get('avg_lux')),
            'low_lux_p5': _optional_scalar(row.get('low_lux_p5')),
            'high_lux_p95': _optional_scalar(row.get('high_lux_p95')),
            'measured_power_w': _optional_scalar(row.get('measured_power_w')),
            'interval_hours': _optional_scalar(row.get('interval_hours')),
            'notes': str(row.get('notes', '')),
        }
        if sample['lux_map'] is None:
            sample['lux_map'] = build_pseudo_lux_map(
                image=sample['image'],
                albedo=sample['albedo'],
                gloss=sample['gloss'],
                depth=sample['depth'],
            )
            sample['notes'] = (sample['notes'] + ' Public-only pseudo-lux supervision.').strip()

        if sample['lux_map'] is not None:
            lux_summary = summarize_lux_map(sample['lux_map'].numpy())
            if sample['avg_lux'] is None:
                sample['avg_lux'] = lux_summary['avg_lux']
            if sample['low_lux_p5'] is None:
                sample['low_lux_p5'] = lux_summary['low_lux_p5']
            if sample['high_lux_p95'] is None:
                sample['high_lux_p95'] = lux_summary['high_lux_p95']

        point_targets_json = row.get('point_targets_json')
        if point_targets_json is None or (isinstance(point_targets_json, float) and np.isnan(point_targets_json)):
            sample['point_lux'] = None
        else:
            path_text = str(point_targets_json).strip()
            sample['point_lux'] = load_point_target_values(path_text) if path_text else None

        if sample['point_lux'] is None and sample['lux_map'] is not None:
            sample['point_lux'] = sample_values_at_points(
                sample['lux_map'],
                build_canonical_point_targets(FIXTURE_COUNT, floor_y=FLOOR_Y, start_x=POINT_START_X, end_x=POINT_END_X),
            )
        return sample


def _collate_optional_dense(samples: Sequence[dict[str, Any]], key: str) -> tuple[torch.Tensor | None, torch.Tensor | None]:
    available = [sample[key] for sample in samples if sample.get(key) is not None]
    if not available:
        return None, None
    template = available[0]
    stacked, mask = [], []
    for sample in samples:
        value = sample.get(key)
        if value is None:
            stacked.append(torch.zeros_like(template))
            mask.append(False)
        else:
            stacked.append(value)
            mask.append(True)
    return torch.stack(stacked, dim=0), torch.tensor(mask, dtype=torch.bool)


def _collate_optional_scalar(samples: Sequence[dict[str, Any]], key: str) -> tuple[torch.Tensor | None, torch.Tensor | None]:
    available = [sample[key] for sample in samples if sample.get(key) is not None]
    if not available:
        return None, None
    values, mask = [], []
    for sample in samples:
        value = sample.get(key)
        if value is None:
            values.append(0.0)
            mask.append(False)
        else:
            values.append(float(value))
            mask.append(True)
    return torch.tensor(values, dtype=torch.float32).unsqueeze(1), torch.tensor(mask, dtype=torch.bool)


def _collate_point_targets(samples: Sequence[dict[str, Any]]) -> tuple[dict[str, torch.Tensor] | None, dict[str, torch.Tensor] | None]:
    available = [sample['point_lux'] for sample in samples if sample.get('point_lux') is not None]
    if not available:
        return None, None
    point_names = sorted({name for point_dict in available for name in point_dict.keys()})
    point_values = {}
    point_masks = {}
    for point_name in point_names:
        values, valid = [], []
        for sample in samples:
            point_dict = sample.get('point_lux')
            if point_dict is None or point_name not in point_dict:
                values.append(0.0)
                valid.append(False)
            else:
                values.append(float(point_dict[point_name]))
                valid.append(True)
        point_values[point_name] = torch.tensor(values, dtype=torch.float32)
        point_masks[point_name] = torch.tensor(valid, dtype=torch.bool)
    return point_values, point_masks


def multitask_collate_fn(samples: Sequence[dict[str, Any]]) -> dict[str, Any]:
    batch = {
        'sample_id': [sample['sample_id'] for sample in samples],
        'dataset_name': [sample['dataset_name'] for sample in samples],
        'split': [sample['split'] for sample in samples],
        'image_path': [sample['image_path'] for sample in samples],
        'notes': [sample['notes'] for sample in samples],
        'image': torch.stack([sample['image'] for sample in samples], dim=0),
    }
    for key in ('depth', 'floor_mask', 'lux_map', 'albedo', 'gloss'):
        values, mask = _collate_optional_dense(samples, key)
        batch[key] = values
        batch[f'{key}_valid_mask'] = mask
    for key in ('avg_lux', 'low_lux_p5', 'high_lux_p95', 'measured_power_w', 'interval_hours'):
        values, mask = _collate_optional_scalar(samples, key)
        batch[key] = values
        batch[f'{key}_valid_mask'] = mask
    point_values, point_masks = _collate_point_targets(samples)
    batch['point_lux'] = point_values
    batch['point_lux_valid_mask'] = point_masks
    return batch


def combine_manifests_for_split(manifests: Mapping[str, pd.DataFrame], split: str) -> pd.DataFrame:
    rows = []
    for manifest in manifests.values():
        split_rows = manifest.loc[manifest['split'].fillna('').astype(str).str.lower() == split.lower()].copy()
        if not split_rows.empty:
            rows.append(split_rows)
    if not rows:
        return pd.DataFrame(columns=MANIFEST_COLUMNS)
    return pd.concat(rows, ignore_index=True)


def build_dataloaders(manifests: Mapping[str, pd.DataFrame], batch_size: int, num_workers: int, image_size: tuple[int, int], seed: int) -> dict[str, DataLoader | None]:
    generator = make_torch_generator(seed)
    dataloaders = {}
    for split_name in ('train', 'val', 'test'):
        split_manifest = combine_manifests_for_split(manifests, split_name)
        if split_manifest.empty:
            dataloaders[split_name] = None
            continue
        dataset = ManifestMultitaskDataset(split_manifest, image_size=image_size)
        dataloaders[split_name] = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=split_name == 'train',
            num_workers=num_workers,
            pin_memory=True,
            collate_fn=multitask_collate_fn,
            worker_init_fn=seed_worker,
            generator=generator,
        )
    return dataloaders


def mae(prediction: torch.Tensor | np.ndarray, target: torch.Tensor | np.ndarray, valid_mask: torch.Tensor | np.ndarray | None = None) -> float:
    pred = prediction.detach().cpu().numpy() if isinstance(prediction, torch.Tensor) else np.asarray(prediction)
    tgt = target.detach().cpu().numpy() if isinstance(target, torch.Tensor) else np.asarray(target)
    err = pred - tgt
    if valid_mask is not None:
        mask = valid_mask.detach().cpu().numpy() if isinstance(valid_mask, torch.Tensor) else np.asarray(valid_mask)
        mask = mask.astype(bool)
        if mask.shape != pred.shape:
            if mask.ndim == 1 and pred.ndim >= 2 and mask.shape[0] == pred.shape[0]:
                mask = mask.reshape((mask.shape[0],) + (1,) * (pred.ndim - 1))
            mask = np.broadcast_to(mask, pred.shape)
        err = err[mask]
    else:
        err = err.reshape(-1)
    if err.size == 0:
        return 0.0
    return float(np.mean(np.abs(err)))


def rmse(prediction: torch.Tensor | np.ndarray, target: torch.Tensor | np.ndarray, valid_mask: torch.Tensor | np.ndarray | None = None) -> float:
    pred = prediction.detach().cpu().numpy() if isinstance(prediction, torch.Tensor) else np.asarray(prediction)
    tgt = target.detach().cpu().numpy() if isinstance(target, torch.Tensor) else np.asarray(target)
    err = pred - tgt
    if valid_mask is not None:
        mask = valid_mask.detach().cpu().numpy() if isinstance(valid_mask, torch.Tensor) else np.asarray(valid_mask)
        mask = mask.astype(bool)
        if mask.shape != pred.shape:
            if mask.ndim == 1 and pred.ndim >= 2 and mask.shape[0] == pred.shape[0]:
                mask = mask.reshape((mask.shape[0],) + (1,) * (pred.ndim - 1))
            mask = np.broadcast_to(mask, pred.shape)
        err = err[mask]
    else:
        err = err.reshape(-1)
    if err.size == 0:
        return 0.0
    return float(np.sqrt(np.mean(np.square(err))))


def pointwise_lux_error(prediction: Mapping[str, torch.Tensor], target: Mapping[str, torch.Tensor] | None, valid_mask: Mapping[str, torch.Tensor] | None = None) -> float | None:
    if target is None:
        return None
    shared = [name for name in prediction.keys() if name in target]
    if not shared:
        return None
    errors = []
    for name in shared:
        errors.append(mae(prediction[name], target[name], None if valid_mask is None else valid_mask.get(name)))
    return float(np.mean(errors)) if errors else None


def log_lux_smooth_l1_loss(prediction: torch.Tensor, target: torch.Tensor | None, valid_mask: torch.Tensor | None = None) -> torch.Tensor:
    if target is None:
        return prediction.new_zeros(())
    pred = torch.log1p(torch.clamp(prediction, min=0.0))
    tgt = torch.log1p(torch.clamp(target, min=0.0))
    loss = F.smooth_l1_loss(pred, tgt, reduction='none')
    if valid_mask is None:
        return loss.mean()
    mask = valid_mask.view(-1, 1, 1, 1).to(loss.device)
    denom = mask.sum().clamp_min(1)
    return (loss * mask).sum() / denom


def masked_smooth_l1_loss(prediction: torch.Tensor, target: torch.Tensor | None, valid_mask: torch.Tensor | None = None) -> torch.Tensor:
    if target is None:
        return prediction.new_zeros(())
    loss = F.smooth_l1_loss(prediction, target, reduction='none')
    if valid_mask is None:
        return loss.mean()
    view_shape = [valid_mask.shape[0]] + [1] * (loss.ndim - 1)
    mask = valid_mask.view(*view_shape).to(loss.device)
    denom = mask.sum().clamp_min(1)
    return (loss * mask).sum() / denom


def pointwise_lux_loss(prediction: Mapping[str, torch.Tensor], target: Mapping[str, torch.Tensor] | None, valid_mask: Mapping[str, torch.Tensor] | None = None) -> torch.Tensor:
    if target is None:
        first_tensor = next(iter(prediction.values()))
        return first_tensor.new_zeros(())
    losses = []
    for name, pred in prediction.items():
        if name not in target:
            continue
        point_loss = F.smooth_l1_loss(pred, target[name].to(pred.device), reduction='none')
        point_mask = None if valid_mask is None else valid_mask.get(name)
        if point_mask is not None:
            mask = point_mask.to(pred.device)
            if mask.sum() == 0:
                continue
            point_loss = (point_loss * mask) / mask.sum().clamp_min(1)
            losses.append(point_loss.sum())
        else:
            losses.append(point_loss.mean())
    if not losses:
        first_tensor = next(iter(prediction.values()))
        return first_tensor.new_zeros(())
    return torch.stack(losses).mean()


def segmentation_loss(logits: torch.Tensor, target: torch.Tensor | None, valid_mask: torch.Tensor | None = None) -> torch.Tensor:
    if target is None:
        return logits.new_zeros(())
    loss = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    if valid_mask is None:
        return loss.mean()
    mask = valid_mask.view(-1, 1, 1, 1).to(loss.device)
    denom = mask.sum().clamp_min(1)
    return (loss * mask).sum() / denom


def heteroscedastic_l1_loss(prediction: torch.Tensor, target: torch.Tensor | None, uncertainty: torch.Tensor, valid_mask: torch.Tensor | None = None) -> torch.Tensor:
    if target is None:
        return prediction.new_zeros(())
    loss = torch.exp(-uncertainty) * torch.abs(prediction - target) + uncertainty
    if valid_mask is None:
        return loss.mean()
    mask = valid_mask.view(-1, 1, 1, 1).to(loss.device)
    denom = mask.sum().clamp_min(1)
    return (loss * mask).sum() / denom


def uncertainty_regularization_loss(uncertainty: torch.Tensor, valid_mask: torch.Tensor | None = None, weight: float = 0.05) -> torch.Tensor:
    loss = weight * uncertainty
    if valid_mask is None:
        return loss.mean()
    mask = valid_mask.view(-1, 1, 1, 1).to(loss.device)
    denom = mask.sum().clamp_min(1)
    return (loss * mask).sum() / denom


def carbon_interval_loss(predicted_power_w: torch.Tensor, target_carbon: torch.Tensor | None, carbon_factor_kg_per_kwh: float, interval_hours: float | torch.Tensor, valid_mask: torch.Tensor | None = None) -> torch.Tensor:
    if target_carbon is None:
        return predicted_power_w.new_zeros(())
    interval_hours_tensor = interval_hours if isinstance(interval_hours, torch.Tensor) else torch.full_like(predicted_power_w, float(interval_hours))
    predicted_carbon = (predicted_power_w / 1000.0) * interval_hours_tensor * carbon_factor_kg_per_kwh
    return masked_smooth_l1_loss(predicted_carbon, target_carbon, valid_mask=valid_mask)


def _move_to_device(value: Any, device: torch.device) -> Any:
    if value is None:
        return None
    if isinstance(value, torch.Tensor):
        return value.to(device)
    if isinstance(value, dict):
        return {key: _move_to_device(item, device) for key, item in value.items()}
    return value


def _dataset_task_mask(dataset_names: Sequence[str], task_name: str, device: torch.device) -> torch.Tensor:
    return torch.tensor([task_name in DATASET_SUPERVISION_RULES.get(name, set()) for name in dataset_names], dtype=torch.bool, device=device)


def _combine_sample_masks(availability_mask: torch.Tensor | None, dataset_mask: torch.Tensor) -> torch.Tensor | None:
    if availability_mask is None:
        return None
    return availability_mask.to(dataset_mask.device) & dataset_mask


def _combine_point_masks(point_masks: dict[str, torch.Tensor] | None, dataset_mask: torch.Tensor) -> dict[str, torch.Tensor] | None:
    if point_masks is None:
        return None
    return {name: mask.to(dataset_mask.device) & dataset_mask for name, mask in point_masks.items()}


def multitask_metrics(outputs: Mapping[str, Any], batch: Mapping[str, Any]) -> dict[str, float]:
    metrics = {}
    if batch.get('lux_map') is not None:
        metrics['lux_mae'] = mae(outputs['lux_map'], batch['lux_map'], batch.get('lux_map_valid_mask'))
        metrics['lux_rmse'] = rmse(outputs['lux_map'], batch['lux_map'], batch.get('lux_map_valid_mask'))
    if batch.get('avg_lux') is not None:
        metrics['avg_lux_error'] = mae(outputs['avg_lux'], batch['avg_lux'], batch.get('avg_lux_valid_mask'))
    if batch.get('low_lux_p5') is not None:
        metrics['p5_error'] = mae(outputs['low_lux_p5'], batch['low_lux_p5'], batch.get('low_lux_p5_valid_mask'))
    if batch.get('high_lux_p95') is not None:
        metrics['p95_error'] = mae(outputs['high_lux_p95'], batch['high_lux_p95'], batch.get('high_lux_p95_valid_mask'))
    point_error = pointwise_lux_error(outputs['point_lux'], batch.get('point_lux'), batch.get('point_lux_valid_mask'))
    if point_error is not None:
        metrics['pointwise_lux_error'] = point_error
    return metrics


def _tensor_batch_to_cpu_dict(point_dict: Mapping[str, torch.Tensor], index: int) -> dict[str, float]:
    return {name: float(values[index].detach().cpu().item()) for name, values in point_dict.items()}


@dataclass
class EpochResult:
    summary: dict[str, float]
    visual_examples: list[dict[str, Any]]
    point_reports: list[dict[str, Any]]


def compute_multitask_loss(outputs: Mapping[str, Any], batch: Mapping[str, Any]) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    device = outputs['lux_map'].device
    dataset_names = batch['dataset_name']
    depth_dataset_mask = _dataset_task_mask(dataset_names, 'depth', device)
    floor_dataset_mask = _dataset_task_mask(dataset_names, 'floor', device)
    lux_dataset_mask = _dataset_task_mask(dataset_names, 'lux', device)
    stats_dataset_mask = _dataset_task_mask(dataset_names, 'stats', device)
    point_dataset_mask = _dataset_task_mask(dataset_names, 'point', device)
    albedo_dataset_mask = _dataset_task_mask(dataset_names, 'albedo', device)
    gloss_dataset_mask = _dataset_task_mask(dataset_names, 'gloss', device)
    power_dataset_mask = _dataset_task_mask(dataset_names, 'power', device)
    carbon_dataset_mask = _dataset_task_mask(dataset_names, 'carbon', device)

    loss_breakdown = {
        'depth': log_lux_smooth_l1_loss(outputs['depth_pred'], batch['depth'], valid_mask=_combine_sample_masks(batch['depth_valid_mask'], depth_dataset_mask)),
        'lux_map': log_lux_smooth_l1_loss(outputs['lux_map'], batch['lux_map'], valid_mask=_combine_sample_masks(batch['lux_map_valid_mask'], lux_dataset_mask)),
        'avg_lux': masked_smooth_l1_loss(outputs['avg_lux'], batch['avg_lux'], valid_mask=_combine_sample_masks(batch['avg_lux_valid_mask'], stats_dataset_mask)),
        'p5_lux': masked_smooth_l1_loss(outputs['low_lux_p5'], batch['low_lux_p5'], valid_mask=_combine_sample_masks(batch['low_lux_p5_valid_mask'], stats_dataset_mask)),
        'p95_lux': masked_smooth_l1_loss(outputs['high_lux_p95'], batch['high_lux_p95'], valid_mask=_combine_sample_masks(batch['high_lux_p95_valid_mask'], stats_dataset_mask)),
        'point_lux': pointwise_lux_loss(outputs['point_lux'], batch['point_lux'], valid_mask=_combine_point_masks(batch['point_lux_valid_mask'], point_dataset_mask)),
        'floor_mask': segmentation_loss(outputs['floor_logits'], batch['floor_mask'], valid_mask=_combine_sample_masks(batch['floor_mask_valid_mask'], floor_dataset_mask)),
        'albedo': masked_smooth_l1_loss(outputs['albedo_pred'], batch['albedo'], valid_mask=_combine_sample_masks(batch['albedo_valid_mask'], albedo_dataset_mask)),
        'gloss': masked_smooth_l1_loss(outputs['gloss_pred'], batch['gloss'], valid_mask=_combine_sample_masks(batch['gloss_valid_mask'], gloss_dataset_mask)),
        'uncertainty': heteroscedastic_l1_loss(outputs['lux_map'], batch['lux_map'], outputs['uncertainty_map'], valid_mask=_combine_sample_masks(batch['lux_map_valid_mask'], lux_dataset_mask))
            + uncertainty_regularization_loss(outputs['uncertainty_map'], valid_mask=_combine_sample_masks(batch['lux_map_valid_mask'], lux_dataset_mask)),
        'power': masked_smooth_l1_loss(outputs['estimated_power_w'], batch['measured_power_w'], valid_mask=_combine_sample_masks(batch['measured_power_w_valid_mask'], power_dataset_mask)),
    }

    if batch['measured_power_w'] is not None:
        interval_hours = batch['interval_hours'] if batch['interval_hours'] is not None else torch.full_like(batch['measured_power_w'], INTERVAL_HOURS)
        target_carbon = batch['measured_power_w'] / 1000.0 * interval_hours * CARBON_FACTOR_KG_PER_KWH
    else:
        interval_hours = INTERVAL_HOURS
        target_carbon = None
    carbon_mask = None
    if batch['measured_power_w_valid_mask'] is not None:
        interval_mask = batch['interval_hours_valid_mask'] if batch['interval_hours_valid_mask'] is not None else batch['measured_power_w_valid_mask']
        carbon_mask = _combine_sample_masks(batch['measured_power_w_valid_mask'] & interval_mask, carbon_dataset_mask)

    loss_breakdown['carbon'] = carbon_interval_loss(outputs['estimated_power_w'], target_carbon, CARBON_FACTOR_KG_PER_KWH, interval_hours, valid_mask=carbon_mask)

    weights = {
        'depth': 0.2,
        'lux_map': 1.0,
        'avg_lux': 0.2,
        'p5_lux': 0.15,
        'p95_lux': 0.15,
        'point_lux': 0.2,
        'floor_mask': 0.3,
        'albedo': 0.2,
        'gloss': 0.1,
        'uncertainty': 0.1,
        'power': 0.05,
        'carbon': 0.1,
    }
    total_loss = outputs['lux_map'].new_zeros(())
    for name, value in loss_breakdown.items():
        total_loss = total_loss + weights.get(name, 1.0) * value
    return total_loss, loss_breakdown


def run_epoch(model: nn.Module, dataloader: DataLoader | None, device: torch.device, optimizer: torch.optim.Optimizer | None = None, scaler: torch.cuda.amp.GradScaler | None = None, amp_enabled: bool = False, max_visualization_examples: int = 4, gradient_clip_norm: float = 1.0) -> EpochResult:
    if dataloader is None:
        return EpochResult(summary={}, visual_examples=[], point_reports=[])
    is_training = optimizer is not None
    model.train(is_training)
    device_type = 'cuda' if device.type == 'cuda' else 'cpu'
    loss_sums = {'total_loss': 0.0}
    metric_values: dict[str, list[float]] = {}
    visual_examples, point_reports = [], []

    progress = tqdm(dataloader, total=len(dataloader), desc='train' if is_training else 'eval', leave=False)
    for batch_index, batch in enumerate(progress, start=1):
        batch = {key: _move_to_device(value, device) for key, value in batch.items()}
        image_tensor = batch['image']
        autocast_enabled = amp_enabled and device_type == 'cuda'
        with torch.set_grad_enabled(is_training):
            with torch.autocast(device_type=device_type, enabled=autocast_enabled):
                outputs = model(image_tensor)
                total_loss, loss_breakdown = compute_multitask_loss(outputs, batch)
        if is_training:
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None and autocast_enabled:
                scaler.scale(total_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_norm)
                optimizer.step()
        loss_sums['total_loss'] += float(total_loss.detach().cpu())
        for loss_name, loss_value in loss_breakdown.items():
            loss_sums.setdefault(loss_name, 0.0)
            loss_sums[loss_name] += float(loss_value.detach().cpu())
        batch_metrics = multitask_metrics(outputs, batch)
        for metric_name, metric_value in batch_metrics.items():
            metric_values.setdefault(metric_name, []).append(metric_value)
        progress.set_postfix(total_loss=f"{loss_sums['total_loss'] / batch_index:.4f}")
        if len(visual_examples) < max_visualization_examples:
            sample_index = 0
            visual_examples.append(
                {
                    'sample_id': batch['sample_id'][sample_index],
                    'dataset_name': batch['dataset_name'][sample_index],
                    'image': batch['image'][sample_index].detach().cpu(),
                    'lux_map_pred': outputs['lux_map'][sample_index].detach().cpu(),
                    'floor_mask_pred': outputs['floor_mask_pred'][sample_index].detach().cpu(),
                    'albedo_pred': outputs['albedo_pred'][sample_index].detach().cpu(),
                    'gloss_pred': outputs['gloss_pred'][sample_index].detach().cpu(),
                    'point_targets': outputs['point_targets'],
                    'point_lux_pred': _tensor_batch_to_cpu_dict(outputs['point_lux'], sample_index),
                    'point_lux_target': _tensor_batch_to_cpu_dict(batch['point_lux'], sample_index) if batch.get('point_lux') is not None else None,
                }
            )
        if batch.get('point_lux') is not None and len(point_reports) < max_visualization_examples:
            sample_index = 0
            point_reports.append(
                {
                    'sample_id': batch['sample_id'][sample_index],
                    'dataset_name': batch['dataset_name'][sample_index],
                    'predicted_points': _tensor_batch_to_cpu_dict(outputs['point_lux'], sample_index),
                    'target_points': _tensor_batch_to_cpu_dict(batch['point_lux'], sample_index),
                }
            )
    batch_count = max(len(dataloader), 1)
    summary = {name: value / batch_count for name, value in loss_sums.items()}
    for metric_name, values in metric_values.items():
        summary[metric_name] = float(np.mean(values)) if values else 0.0
    return EpochResult(summary=summary, visual_examples=visual_examples, point_reports=point_reports)


## 8. Model Definition

The model is a shared encoder-decoder that predicts dense lux, floor mask, albedo proxy, gloss proxy, uncertainty, scalar lux stats, and point-wise fixture values.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


def _adapt_resnet_input_conv(conv: nn.Conv2d, in_channels: int) -> nn.Conv2d:
    if conv.in_channels == in_channels:
        return conv
    adapted = nn.Conv2d(
        in_channels,
        conv.out_channels,
        kernel_size=conv.kernel_size,
        stride=conv.stride,
        padding=conv.padding,
        bias=conv.bias is not None,
    )
    with torch.no_grad():
        original_weight = conv.weight.data
        adapted.weight.zero_()
        shared_channels = min(in_channels, conv.in_channels)
        adapted.weight[:, :shared_channels] = original_weight[:, :shared_channels]
        if in_channels > conv.in_channels:
            mean_channel = original_weight.mean(dim=1, keepdim=True)
            for channel_index in range(conv.in_channels, in_channels):
                adapted.weight[:, channel_index: channel_index + 1] = mean_channel
    return adapted


class ResNet18Backbone(nn.Module):
    def __init__(self, in_channels: int = 3, pretrained: bool = False) -> None:
        super().__init__()
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        encoder = resnet18(weights=weights)
        encoder.conv1 = _adapt_resnet_input_conv(encoder.conv1, in_channels)
        self.conv1 = encoder.conv1
        self.bn1 = encoder.bn1
        self.relu = encoder.relu
        self.maxpool = encoder.maxpool
        self.layer1 = encoder.layer1
        self.layer2 = encoder.layer2
        self.layer3 = encoder.layer3
        self.layer4 = encoder.layer4
        self.feature_channels = (64, 64, 128, 256, 512)
        self.in_channels = in_channels

    def forward(self, x: torch.Tensor) -> list[torch.Tensor]:
        stem = self.relu(self.bn1(self.conv1(x)))
        layer1 = self.layer1(self.maxpool(stem))
        layer2 = self.layer2(layer1)
        layer3 = self.layer3(layer2)
        layer4 = self.layer4(layer3)
        return [stem, layer1, layer2, layer3, layer4]


class UpBlock(nn.Module):
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int) -> None:
        super().__init__()
        self.projection = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.fuse = ConvBlock(out_channels + skip_channels, out_channels)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        x = self.projection(x)
        return self.fuse(torch.cat([x, skip], dim=1))


class SegmentationHead(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 1) -> None:
        super().__init__()
        self.head = nn.Sequential(
            ConvBlock(in_channels, in_channels),
            nn.Conv2d(in_channels, out_channels, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(x)


class DensePredictionHead(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.head = nn.Sequential(
            ConvBlock(in_channels, in_channels),
            nn.Conv2d(in_channels, out_channels, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(x)


class ScalarRegressionHead(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 1) -> None:
        super().__init__()
        hidden = max(in_channels // 2, 32)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, out_channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.projection(self.pool(x))


class LuxStatisticsHead(nn.Module):
    def __init__(self, in_channels: int) -> None:
        super().__init__()
        self.avg_head = ScalarRegressionHead(in_channels, 1)
        self.p5_head = ScalarRegressionHead(in_channels, 1)
        self.p95_head = ScalarRegressionHead(in_channels, 1)
        self.softplus = nn.Softplus(beta=1.0)

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        avg = self.softplus(self.avg_head(x))
        p5 = self.softplus(self.p5_head(x))
        p95 = self.softplus(self.p95_head(x))
        return {'avg_lux': avg, 'low_lux_p5': p5, 'high_lux_p95': torch.maximum(p95, p5)}


class HallwayMultitaskUNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        input_channels = 3 + (2 if USE_COORD_CHANNELS else 0)
        self.backbone = ResNet18Backbone(in_channels=input_channels, pretrained=USE_PRETRAINED_BACKBONE)
        encoder_channels = list(self.backbone.feature_channels)
        decoder_channels = [256, 128, 64, 64]

        self.bottleneck = ConvBlock(encoder_channels[-1], decoder_channels[0])
        self.decoder_blocks = nn.ModuleList([
            UpBlock(decoder_channels[0], encoder_channels[-2], decoder_channels[0]),
            UpBlock(decoder_channels[0], encoder_channels[-3], decoder_channels[1]),
            UpBlock(decoder_channels[1], encoder_channels[-4], decoder_channels[2]),
            UpBlock(decoder_channels[2], encoder_channels[-5], decoder_channels[3]),
        ])
        self.final_refine = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBlock(decoder_channels[-1], decoder_channels[-1]),
        )

        head_channels = decoder_channels[-1]
        self.floor_head = SegmentationHead(head_channels, 1)
        self.lux_head = DensePredictionHead(head_channels, 1)
        self.albedo_head = DensePredictionHead(head_channels, 3)
        self.gloss_head = DensePredictionHead(head_channels, 1)
        self.depth_head = DensePredictionHead(head_channels, 1)
        self.uncertainty_head = DensePredictionHead(head_channels, 1)
        self.stats_head = LuxStatisticsHead(head_channels)
        self.power_head = ScalarRegressionHead(head_channels, 1)
        self.softplus = nn.Softplus(beta=1.0)

    def _prepare_inputs(self, image: torch.Tensor) -> torch.Tensor:
        if not USE_COORD_CHANNELS:
            return image
        batch_size, _, height, width = image.shape
        coords = expand_coordinate_channels(batch_size, height, width, device=image.device, dtype=image.dtype)
        return torch.cat([image, coords], dim=1)

    def _decode(self, features: list[torch.Tensor], output_size: tuple[int, int]) -> torch.Tensor:
        stem, layer1, layer2, layer3, layer4 = features
        decoder = self.bottleneck(layer4)
        decoder = self.decoder_blocks[0](decoder, layer3)
        decoder = self.decoder_blocks[1](decoder, layer2)
        decoder = self.decoder_blocks[2](decoder, layer1)
        decoder = self.decoder_blocks[3](decoder, stem)
        decoder = self.final_refine(decoder)
        if decoder.shape[-2:] != output_size:
            decoder = F.interpolate(decoder, size=output_size, mode='bilinear', align_corners=False)
        return decoder

    def forward(self, image: torch.Tensor, point_targets: list[PointTarget] | None = None) -> dict[str, Any]:
        prepared = self._prepare_inputs(image)
        features = self.backbone(prepared)
        decoder_features = self._decode(features, output_size=image.shape[-2:])
        floor_logits = self.floor_head(decoder_features)
        floor_mask_pred = torch.sigmoid(floor_logits)
        lux_map = self.softplus(self.lux_head(decoder_features))
        albedo_pred = torch.sigmoid(self.albedo_head(decoder_features))
        gloss_pred = torch.sigmoid(self.gloss_head(decoder_features))
        depth_pred = self.softplus(self.depth_head(decoder_features))
        uncertainty_map = self.softplus(self.uncertainty_head(decoder_features)) + 1e-6
        statistics = self.stats_head(decoder_features)
        estimated_power_w = self.softplus(self.power_head(decoder_features))
        effective_points = point_targets or build_canonical_point_targets(FIXTURE_COUNT, floor_y=FLOOR_Y, start_x=POINT_START_X, end_x=POINT_END_X)
        point_lux = sample_point_values_batch(lux_map, effective_points)
        return {
            'lux_map': lux_map,
            'avg_lux': statistics['avg_lux'],
            'low_lux_p5': statistics['low_lux_p5'],
            'high_lux_p95': statistics['high_lux_p95'],
            'point_lux': point_lux,
            'point_targets': effective_points,
            'floor_logits': floor_logits,
            'floor_mask_pred': floor_mask_pred,
            'albedo_pred': albedo_pred,
            'gloss_pred': gloss_pred,
            'depth_pred': depth_pred,
            'uncertainty_map': uncertainty_map,
            'estimated_power_w': estimated_power_w,
        }


## 9. Build Dataloaders And Initialize Training State

This creates the dataloaders, model, optimizer, scheduler, and checkpoint state.


In [ ]:
set_seed(SEED)
DATALOADERS = build_dataloaders(MANIFESTS, BATCH_SIZE, NUM_WORKERS, IMAGE_SIZE, SEED)
train_loader = DATALOADERS['train']
val_loader = DATALOADERS['val']
test_loader = DATALOADERS['test']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
amp_enabled = AMP_ENABLED and device.type == 'cuda'
model = HallwayMultitaskUNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(TRAIN_EPOCHS, 1))
scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

HISTORY = {'train': [], 'val': [], 'test': []}
START_EPOCH = 0
BEST_SCORE = float('inf')
TRAIN_RESULT = None
VAL_RESULT = None
TEST_RESULT = None

if RESUME_CHECKPOINT_PATH:
    checkpoint = load_checkpoint(RESUME_CHECKPOINT_PATH, model, optimizer=optimizer, scheduler=scheduler, scaler=scaler, map_location=device)
    HISTORY = checkpoint.get('history', HISTORY)
    START_EPOCH = int(checkpoint.get('epoch', -1)) + 1
    BEST_SCORE = float(checkpoint.get('extra_state', {}).get('best_score', BEST_SCORE))

batch_rows = []
for split_name, loader in [('train', train_loader), ('val', val_loader), ('test', test_loader)]:
    batch_rows.append({'split': split_name, 'available': loader is not None, 'num_batches': 0 if loader is None else len(loader)})
display(pd.DataFrame(batch_rows))

if train_loader is not None:
    first_batch = next(iter(train_loader))
    batch_summary = {
        'image_shape': tuple(first_batch['image'].shape),
        'datasets_in_batch': list(first_batch['dataset_name']),
        'has_depth': first_batch['depth'] is not None,
        'has_lux_map': first_batch['lux_map'] is not None,
        'has_floor_mask': first_batch['floor_mask'] is not None,
        'has_albedo': first_batch['albedo'] is not None,
        'has_gloss': first_batch['gloss'] is not None,
        'has_point_targets': first_batch['point_lux'] is not None,
    }
    batch_summary


## 10. Training

Run this cell to train the prototype model.


In [ ]:
if RUN_TRAINING:
    if train_loader is None:
        raise RuntimeError('No training data is available. You need at least one dataset with train rows.')
    for epoch in range(START_EPOCH, TRAIN_EPOCHS):
        TRAIN_RESULT = run_epoch(
            model=model,
            dataloader=train_loader,
            device=device,
            optimizer=optimizer,
            scaler=scaler,
            amp_enabled=amp_enabled,
            max_visualization_examples=MAX_VIS_EXAMPLES,
            gradient_clip_norm=GRAD_CLIP_NORM,
        )
        train_summary = {'epoch': epoch + 1, **TRAIN_RESULT.summary}
        HISTORY['train'].append(train_summary)

        if val_loader is not None:
            VAL_RESULT = run_epoch(
                model=model,
                dataloader=val_loader,
                device=device,
                optimizer=None,
                scaler=None,
                amp_enabled=amp_enabled,
                max_visualization_examples=MAX_VIS_EXAMPLES,
                gradient_clip_norm=GRAD_CLIP_NORM,
            )
            val_summary = {'epoch': epoch + 1, **VAL_RESULT.summary}
            HISTORY['val'].append(val_summary)
            monitored_score = float(val_summary.get('total_loss', train_summary.get('total_loss', 0.0)))
        else:
            monitored_score = float(train_summary.get('total_loss', 0.0))

        scheduler.step()
        save_checkpoint(model, optimizer, epoch, LAST_MODEL_PATH, scheduler=scheduler, scaler=scaler, history=HISTORY, extra_state={'best_score': BEST_SCORE})
        if monitored_score <= BEST_SCORE:
            BEST_SCORE = monitored_score
            save_checkpoint(model, optimizer, epoch, BEST_MODEL_PATH, scheduler=scheduler, scaler=scaler, history=HISTORY, extra_state={'best_score': BEST_SCORE})
        save_training_history(HISTORY, HISTORY_PATH)
        print(f'Epoch {epoch + 1}/{TRAIN_EPOCHS} | train_total_loss={train_summary.get("total_loss", 0.0):.4f} | best_score={BEST_SCORE:.4f}')
else:
    print('RUN_TRAINING is False.')


## 11. Validation And Testing

This computes split-level metrics after training.


In [ ]:
if RUN_VALIDATION:
    if val_loader is None:
        raise RuntimeError('No validation split is available.')
    VAL_RESULT = run_epoch(model, val_loader, device, optimizer=None, scaler=None, amp_enabled=amp_enabled, max_visualization_examples=MAX_VIS_EXAMPLES, gradient_clip_norm=GRAD_CLIP_NORM)
    display(pd.DataFrame([VAL_RESULT.summary]))
else:
    print('RUN_VALIDATION is False.')

if RUN_TEST:
    if test_loader is None:
        raise RuntimeError('No test split is available.')
    TEST_RESULT = run_epoch(model, test_loader, device, optimizer=None, scaler=None, amp_enabled=amp_enabled, max_visualization_examples=MAX_VIS_EXAMPLES, gradient_clip_norm=GRAD_CLIP_NORM)
    display(pd.DataFrame([TEST_RESULT.summary]))
else:
    print('RUN_TEST is False.')


## 12. Visualize Latest Prediction

Shows the latest predicted lux map, floor mask, albedo proxy, and point-wise hallway lux locations.


In [ ]:
def overlay_points(image: torch.Tensor | np.ndarray, points: Sequence[PointTarget], title: str = 'Point Targets', point_values: Mapping[str, float] | None = None, ax: plt.Axes | None = None) -> plt.Axes:
    image_np = prepare_display_image(image, denormalize=False)
    target_ax = ax or plt.figure(figsize=(6, 4)).add_subplot(111)
    target_ax.imshow(image_np, cmap='viridis' if image_np.ndim == 2 else None)
    height, width = image_np.shape[:2]
    for point in points:
        x_pixel = point.x * (width - 1)
        y_pixel = point.y * (height - 1)
        target_ax.scatter(x_pixel, y_pixel, s=45, c='white', edgecolors='black')
        label = point.name if point_values is None or point.name not in point_values else f"{point.name}\n{point_values[point.name]:.1f}"
        target_ax.text(x_pixel + 4, y_pixel - 4, label, fontsize=8, color='white')
    target_ax.set_title(title)
    target_ax.axis('off')
    return target_ax


def save_prediction_overview(example: dict[str, Any], path: str | Path) -> Path:
    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    axes[0].imshow(prepare_display_image(example['image']))
    axes[0].set_title('Input')
    axes[0].axis('off')
    axes[1].imshow(prepare_display_image(example['lux_map_pred'], denormalize=False), cmap='viridis')
    axes[1].set_title('Predicted Lux')
    axes[1].axis('off')
    axes[2].imshow(prepare_display_image(example['floor_mask_pred'], denormalize=False), cmap='gray')
    axes[2].set_title('Predicted Floor')
    axes[2].axis('off')
    axes[3].imshow(prepare_display_image(example['albedo_pred']))
    axes[3].set_title('Albedo Proxy')
    axes[3].axis('off')
    overlay_points(example['lux_map_pred'], example['point_targets'], title='Point-wise Lux', point_values=example['point_lux_pred'], ax=axes[4])
    fig.tight_layout()
    return save_figure(fig, path)

RESULT_FOR_VIS = TEST_RESULT if TEST_RESULT is not None else VAL_RESULT if VAL_RESULT is not None else TRAIN_RESULT
if RESULT_FOR_VIS is None or not RESULT_FOR_VIS.visual_examples:
    print('No visual examples are available yet.')
else:
    example = RESULT_FOR_VIS.visual_examples[0]
    figure_path = VIS_DIR / f"{example['sample_id']}_overview.png"
    save_prediction_overview(example, figure_path)
    print(figure_path)
    display(Image.open(figure_path))


## 13. Point-Wise Hallway Report

Shows one sample report with under-fixture and between-fixture lux values plus average/high/low/discrepancy.


In [ ]:
RESULT_FOR_REPORT = TEST_RESULT if TEST_RESULT is not None else VAL_RESULT if VAL_RESULT is not None else TRAIN_RESULT
if RESULT_FOR_REPORT is None or not RESULT_FOR_REPORT.visual_examples:
    print('No results are available yet.')
else:
    example = RESULT_FOR_REPORT.visual_examples[0]
    lux_summary = summarize_lux_map(example['lux_map_pred'])
    report = {
        'sample_id': example['sample_id'],
        'dataset_name': example['dataset_name'],
        'avg_lux': lux_summary['avg_lux'],
        'low_lux_p5': lux_summary['low_lux_p5'],
        'high_lux_p95': lux_summary['high_lux_p95'],
        'lux_discrepancy': lux_summary['lux_discrepancy'],
        **example['point_lux_pred'],
    }
    display(pd.DataFrame([report]))
    if example['point_lux_target'] is not None:
        display(pd.DataFrame([example['point_lux_target']]))


## 14. Client CSV Export And Summary UI

Exports a CSV for the evaluated hallway outputs and renders a lightweight HTML summary card that can be shown to a client.


In [ ]:
from html import escape
from IPython.display import HTML


def estimate_power_from_illuminance(avg_lux: float, floor_area_m2: float, luminous_efficacy_lm_per_w: float, utilization_factor: float, maintenance_factor: float) -> float:
    utilization_factor = max(utilization_factor, 1e-3)
    maintenance_factor = max(maintenance_factor, 1e-3)
    luminous_efficacy_lm_per_w = max(luminous_efficacy_lm_per_w, 1e-3)
    delivered_lumens = avg_lux * floor_area_m2
    required_source_lumens = delivered_lumens / (utilization_factor * maintenance_factor)
    return required_source_lumens / luminous_efficacy_lm_per_w


def flatten_point_report(report: dict[str, Any]) -> dict[str, Any]:
    avg_lux = float(report['avg_lux'])
    estimated_power_w = estimate_power_from_illuminance(
        avg_lux,
        HALLWAY_FLOOR_AREA_M2,
        LUMINOUS_EFFICACY_LM_PER_W,
        UTILIZATION_FACTOR,
        MAINTENANCE_FACTOR,
    )
    interval_energy_kwh = estimated_power_w / 1000.0 * (PHOTO_INTERVAL_MINUTES / 60.0)
    interval_carbon_kg = interval_energy_kwh * GRID_EMISSION_FACTOR_KG_PER_KWH
    row = {
        'sample_id': report['sample_id'],
        'dataset_name': report['dataset_name'],
        'avg_lux': avg_lux,
        'low_lux_p5': report['low_lux_p5'],
        'high_lux_p95': report['high_lux_p95'],
        'lux_discrepancy': report['lux_discrepancy'],
        'estimated_power_w': estimated_power_w,
        'interval_energy_kwh': interval_energy_kwh,
        'interval_carbon_kg': interval_carbon_kg,
    }
    for name, value in sorted(report.get('predicted_points', {}).items()):
        row[name] = value
    target_points = report.get('target_points') or {}
    for name, value in sorted(target_points.items()):
        row[f'target_{name}'] = value
    return row


def export_point_reports_csv(result: EpochResult | None, path: str | Path) -> Path | None:
    if result is None or not result.point_reports:
        print('No point reports are available yet.')
        return None
    rows = [flatten_point_report(report) for report in result.point_reports]
    dataframe = pd.DataFrame(rows)
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(output_path, index=False)
    return output_path


def render_client_summary_html(report: dict[str, Any], title: str = 'Hallway Lighting Summary') -> str:
    flat = flatten_point_report(report)
    point_rows = ''.join(
        f"<tr><td style='padding:6px 10px;border-bottom:1px solid #e5e7eb'>{escape(name)}</td><td style='padding:6px 10px;border-bottom:1px solid #e5e7eb;text-align:right'>{value:.1f}</td></tr>"
        for name, value in flat.items() if name.startswith('under_fixture_') or name.startswith('between_fixture_')
    )
    return f"""
    <div style='font-family:Arial,sans-serif;max-width:980px;margin:12px 0;padding:20px;border:1px solid #d1d5db;border-radius:16px;background:#ffffff'>
      <h2 style='margin:0 0 14px 0;font-size:24px;color:#111827'>{escape(title)}</h2>
      <div style='display:grid;grid-template-columns:repeat(4,minmax(120px,1fr));gap:12px;margin-bottom:18px'>
        <div style='padding:12px;border-radius:12px;background:#eff6ff'><div style='font-size:12px;color:#1d4ed8'>Average Lux</div><div style='font-size:24px;font-weight:700'>{flat['avg_lux']:.1f}</div></div>
        <div style='padding:12px;border-radius:12px;background:#f3f4f6'><div style='font-size:12px;color:#6b7280'>Low Lux P5</div><div style='font-size:24px;font-weight:700'>{flat['low_lux_p5']:.1f}</div></div>
        <div style='padding:12px;border-radius:12px;background:#f3f4f6'><div style='font-size:12px;color:#6b7280'>High Lux P95</div><div style='font-size:24px;font-weight:700'>{flat['high_lux_p95']:.1f}</div></div>
        <div style='padding:12px;border-radius:12px;background:#fef3c7'><div style='font-size:12px;color:#92400e'>Discrepancy</div><div style='font-size:24px;font-weight:700;color:#92400e'>{flat['lux_discrepancy']:.1f}</div></div>
      </div>
      <div style='display:grid;grid-template-columns:repeat(3,minmax(140px,1fr));gap:12px;margin-bottom:18px'>
        <div style='padding:12px;border-radius:12px;background:#ecfccb'><div style='font-size:12px;color:#3f6212'>Estimated Power</div><div style='font-size:22px;font-weight:700'>{flat['estimated_power_w']:.1f} W</div></div>
        <div style='padding:12px;border-radius:12px;background:#ecfeff'><div style='font-size:12px;color:#155e75'>5-Min Energy</div><div style='font-size:22px;font-weight:700'>{flat['interval_energy_kwh']:.4f} kWh</div></div>
        <div style='padding:12px;border-radius:12px;background:#fdf2f8'><div style='font-size:12px;color:#9d174d'>5-Min Carbon</div><div style='font-size:22px;font-weight:700'>{flat['interval_carbon_kg']:.4f} kgCO2e</div></div>
      </div>
      <div style='margin-bottom:10px;font-size:14px;color:#374151'>Sample: <strong>{escape(str(flat['sample_id']))}</strong> | Dataset: <strong>{escape(str(flat['dataset_name']))}</strong></div>
      <table style='width:100%;border-collapse:collapse;font-size:14px'>
        <thead>
          <tr><th style='text-align:left;padding:8px 10px;background:#111827;color:white'>Point</th><th style='text-align:right;padding:8px 10px;background:#111827;color:white'>Predicted Lux</th></tr>
        </thead>
        <tbody>{point_rows}</tbody>
      </table>
    </div>
    """


def save_client_summary_html(report: dict[str, Any], path: str | Path, title: str = 'Hallway Lighting Summary') -> Path:
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(render_client_summary_html(report, title=title), encoding='utf-8')
    return output_path


RESULT_FOR_EXPORT = TEST_RESULT if TEST_RESULT is not None else VAL_RESULT if VAL_RESULT is not None else TRAIN_RESULT
CSV_EXPORT_PATH = REPORT_DIR / 'hallway_client_summary.csv'
HTML_EXPORT_PATH = REPORT_DIR / 'hallway_client_summary.html'

if RESULT_FOR_EXPORT is None or not RESULT_FOR_EXPORT.point_reports:
    print('No exportable hallway reports are available yet.')
else:
    csv_path = export_point_reports_csv(RESULT_FOR_EXPORT, CSV_EXPORT_PATH)
    client_report = RESULT_FOR_EXPORT.point_reports[0]
    html_path = save_client_summary_html(client_report, HTML_EXPORT_PATH)
    display(pd.DataFrame([flatten_point_report(client_report)]))
    display(HTML(render_client_summary_html(client_report)))
    print(csv_path)
    print(html_path)


## 15. Carbon Reporting

This uses a simple photometric prototype formula:

- delivered lumens on floor = `avg_lux * floor_area_m2`
- required source lumens = `delivered lumens / (utilization_factor * maintenance_factor)`
- electrical power = `required source lumens / luminous_efficacy_lm_per_w`
- interval energy = `power * 5 minutes`
- interval carbon = `interval energy * grid emission factor`

These values are configurable and should be calibrated for the actual site.


In [ ]:
RESULT_FOR_CARBON = TEST_RESULT if TEST_RESULT is not None else VAL_RESULT if VAL_RESULT is not None else TRAIN_RESULT
avg_lux_for_carbon = 150.0
if RESULT_FOR_CARBON is not None and RESULT_FOR_CARBON.point_reports:
    avg_lux_for_carbon = float(RESULT_FOR_CARBON.point_reports[0]['avg_lux'])

estimated_power_w = estimate_power_from_illuminance(
    avg_lux_for_carbon,
    HALLWAY_FLOOR_AREA_M2,
    LUMINOUS_EFFICACY_LM_PER_W,
    UTILIZATION_FACTOR,
    MAINTENANCE_FACTOR,
)
interval_energy_kwh = estimated_power_w / 1000.0 * (PHOTO_INTERVAL_MINUTES / 60.0)
interval_carbon_kg = interval_energy_kwh * GRID_EMISSION_FACTOR_KG_PER_KWH

pd.DataFrame([
    {
        'avg_lux': avg_lux_for_carbon,
        'hallway_floor_area_m2': HALLWAY_FLOOR_AREA_M2,
        'luminous_efficacy_lm_per_w': LUMINOUS_EFFICACY_LM_PER_W,
        'utilization_factor': UTILIZATION_FACTOR,
        'maintenance_factor': MAINTENANCE_FACTOR,
        'estimated_power_w': estimated_power_w,
        'interval_minutes': PHOTO_INTERVAL_MINUTES,
        'interval_energy_kwh': interval_energy_kwh,
        'grid_emission_factor_kg_per_kwh': GRID_EMISSION_FACTOR_KG_PER_KWH,
        'interval_carbon_kg': interval_carbon_kg,
    }
])


## 16. Raspberry Pi Capture Policy Prototype

For deployment, the Raspberry Pi should not score a frame if the hallway floor is visibly occluded by a person or moving object. The prototype policy below checks motion over the predicted floor region and defers the next inference if too much floor area is blocked.


In [ ]:
def compute_motion_mask(current_image: Image.Image, previous_image: Image.Image, threshold: float = 18.0) -> np.ndarray:
    current_gray = np.asarray(current_image.convert('L'), dtype=np.float32)
    previous_gray = np.asarray(previous_image.convert('L').resize(current_image.size), dtype=np.float32)
    diff = np.abs(current_gray - previous_gray)
    return diff > threshold


def should_defer_capture(current_image: Image.Image, previous_image: Image.Image | None, predicted_floor_mask: np.ndarray | None = None, max_floor_occlusion_ratio: float = MAX_FLOOR_OCCLUSION_RATIO) -> dict[str, Any]:
    if previous_image is None or predicted_floor_mask is None:
        return {'defer': False, 'reason': 'no_previous_frame_or_floor_mask', 'floor_occlusion_ratio': 0.0}

    motion_mask = compute_motion_mask(current_image, previous_image)
    floor_mask = np.asarray(predicted_floor_mask) > 0.5
    if floor_mask.ndim == 3:
        floor_mask = floor_mask[..., 0]
    floor_mask = floor_mask.astype(bool)
    if floor_mask.sum() == 0:
        return {'defer': False, 'reason': 'empty_floor_mask', 'floor_occlusion_ratio': 0.0}

    floor_occlusion_ratio = float(motion_mask[floor_mask].mean())
    return {
        'defer': floor_occlusion_ratio > max_floor_occlusion_ratio,
        'reason': 'floor_occluded_by_motion' if floor_occlusion_ratio > max_floor_occlusion_ratio else 'frame_accepted',
        'floor_occlusion_ratio': floor_occlusion_ratio,
        'retry_after_seconds': DEFER_SECONDS_ON_OCCLUSION if floor_occlusion_ratio > max_floor_occlusion_ratio else 0,
    }


pd.DataFrame([
    {
        'capture_interval_minutes': PHOTO_INTERVAL_MINUTES,
        'defer_seconds_on_occlusion': DEFER_SECONDS_ON_OCCLUSION,
        'max_defer_seconds': MAX_DEFER_SECONDS,
        'max_floor_occlusion_ratio': MAX_FLOOR_OCCLUSION_RATIO,
        'policy': 'If a person or object blocks more than the allowed floor area, wait and retry before running lux inference.',
    }
])


## 17. Single-Image Inference

Run this after training with a hallway photo to produce JSON, CSV, HTML, and image artifacts.


In [ ]:
@torch.no_grad()
def run_single_image_inference(model: nn.Module, image_path: str | Path, device: torch.device) -> dict[str, Any]:
    transform = build_image_transform(IMAGE_SIZE)
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    outputs = model(image_tensor)
    lux_map = outputs['lux_map'].detach().cpu()
    point_targets = outputs['point_targets']
    point_lux = sample_values_at_points(lux_map, point_targets)
    lux_summary = summarize_lux_map(lux_map)
    avg_lux = float(outputs['avg_lux'].detach().cpu().view(-1)[0])
    low_lux_p5 = float(outputs['low_lux_p5'].detach().cpu().view(-1)[0])
    high_lux_p95 = float(outputs['high_lux_p95'].detach().cpu().view(-1)[0])
    discrepancy = high_lux_p95 - low_lux_p5
    estimated_power_w = estimate_power_from_illuminance(
        avg_lux,
        HALLWAY_FLOOR_AREA_M2,
        LUMINOUS_EFFICACY_LM_PER_W,
        UTILIZATION_FACTOR,
        MAINTENANCE_FACTOR,
    )
    interval_energy_kwh = estimated_power_w / 1000.0 * (PHOTO_INTERVAL_MINUTES / 60.0)
    interval_carbon_kg = interval_energy_kwh * GRID_EMISSION_FACTOR_KG_PER_KWH

    image_stem = Path(image_path).stem
    heatmap_path = INFER_DIR / f'{image_stem}_lux_heatmap.png'
    point_overlay_path = INFER_DIR / f'{image_stem}_point_overlay.png'
    summary_path = INFER_DIR / f'{image_stem}_summary.json'
    summary_csv_path = INFER_DIR / f'{image_stem}_summary.csv'
    summary_html_path = INFER_DIR / f'{image_stem}_summary.html'

    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    ax.imshow(prepare_display_image(lux_map, denormalize=False), cmap='viridis')
    ax.set_title('Predicted Lux Heatmap')
    ax.axis('off')
    save_figure(fig, heatmap_path)

    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    overlay_points(lux_map, point_targets, title='Point-wise Lux', point_values=point_lux, ax=ax)
    save_figure(fig, point_overlay_path)

    flat_report = {
        'sample_id': image_stem,
        'dataset_name': 'single_image_inference',
        'avg_lux': avg_lux,
        'low_lux_p5': low_lux_p5,
        'high_lux_p95': high_lux_p95,
        'lux_discrepancy': discrepancy,
        'estimated_power_w': estimated_power_w,
        'interval_energy_kwh': interval_energy_kwh,
        'interval_carbon_kg': interval_carbon_kg,
        **point_lux,
    }
    pd.DataFrame([flat_report]).to_csv(summary_csv_path, index=False)
    save_client_summary_html(
        {
            'sample_id': image_stem,
            'dataset_name': 'single_image_inference',
            'avg_lux': avg_lux,
            'low_lux_p5': low_lux_p5,
            'high_lux_p95': high_lux_p95,
            'lux_discrepancy': discrepancy,
            'predicted_points': point_lux,
        },
        summary_html_path,
        title='Single Image Hallway Lighting Summary',
    )

    summary = {
        'image_path': str(image_path),
        'avg_lux': avg_lux,
        'low_lux_p5': low_lux_p5,
        'high_lux_p95': high_lux_p95,
        'lux_discrepancy': discrepancy,
        'point_lux': point_lux,
        'estimated_power_w': estimated_power_w,
        'interval_energy_kwh': interval_energy_kwh,
        'interval_carbon_kg': interval_carbon_kg,
        'lux_map_summary': lux_summary,
        'artifacts': {
            'lux_heatmap_image': str(heatmap_path),
            'point_overlay_image': str(point_overlay_path),
            'summary_json': str(summary_path),
            'summary_csv': str(summary_csv_path),
            'summary_html': str(summary_html_path),
        },
    }
    save_json(summary, summary_path)
    return summary


if RUN_SINGLE_IMAGE_INFERENCE:
    if not SINGLE_IMAGE_PATH:
        raise ValueError('Set SINGLE_IMAGE_PATH before running single-image inference.')
    inference_summary = run_single_image_inference(model, SINGLE_IMAGE_PATH, device)
    display(pd.DataFrame([inference_summary]))
    print(json.dumps(inference_summary, indent=2))
else:
    print('RUN_SINGLE_IMAGE_INFERENCE is False.')


## 18. ONNX Export

Exports the model for later Raspberry Pi prototype deployment.


In [ ]:
class OnnxExportWrapper(nn.Module):
    def __init__(self, wrapped_model: nn.Module) -> None:
        super().__init__()
        self.wrapped_model = wrapped_model

    def forward(self, image: torch.Tensor):
        outputs = self.wrapped_model(image)
        return (
            outputs['lux_map'],
            outputs['avg_lux'],
            outputs['low_lux_p5'],
            outputs['high_lux_p95'],
            outputs['floor_mask_pred'],
            outputs['albedo_pred'],
            outputs['gloss_pred'],
            outputs['uncertainty_map'],
            outputs['estimated_power_w'],
        )

if RUN_ONNX_EXPORT:
    onnx_path = Path(ONNX_EXPORT_PATH)
    onnx_path.parent.mkdir(parents=True, exist_ok=True)
    dummy_input = torch.randn(1, 3, IMAGE_SIZE[0], IMAGE_SIZE[1], device=device)
    torch.onnx.export(
        OnnxExportWrapper(model).to(device),
        dummy_input,
        onnx_path,
        input_names=['image'],
        output_names=['lux_map', 'avg_lux', 'low_lux_p5', 'high_lux_p95', 'floor_mask_pred', 'albedo_pred', 'gloss_pred', 'uncertainty_map', 'estimated_power_w'],
        opset_version=17,
        dynamic_axes={'image': {0: 'batch_size'}, 'lux_map': {0: 'batch_size'}},
    )
    metadata_path = onnx_path.with_suffix('.metadata.json')
    save_json(
        {
            'onnx_path': str(onnx_path),
            'input_shape': [1, 3, IMAGE_SIZE[0], IMAGE_SIZE[1]],
            'output_names': ['lux_map', 'avg_lux', 'low_lux_p5', 'high_lux_p95', 'floor_mask_pred', 'albedo_pred', 'gloss_pred', 'uncertainty_map', 'estimated_power_w'],
            'fixture_count': FIXTURE_COUNT,
            'point_targets': [asdict(point) for point in build_canonical_point_targets(FIXTURE_COUNT, floor_y=FLOOR_Y, start_x=POINT_START_X, end_x=POINT_END_X)],
        },
        metadata_path,
    )
    print(onnx_path)
    print(metadata_path)
else:
    print('RUN_ONNX_EXPORT is False.')


## 19. Limitations And Future Improvements

Current limitations:
- The notebook does not use direct hallway lux measurements, so absolute lux can drift from reality.
- Carbon is estimated from predicted illuminance plus configurable lighting assumptions, not from direct power metering.
- Camera exposure, white balance, lens angle, daylight spill, floor gloss, shadows, and unusual fixture types can distort predictions.
- People or carts crossing the floor can temporarily bias reflectance and lux estimates if frame acceptance is not filtered.
- Public datasets do not provide a fully hallway-specific supervision signal for under-fixture and between-fixture lux targets.

High-value improvements for the future:
- collect a small custom hallway dataset with measured lux points and floor masks
- calibrate camera intrinsics, mounting angle, and scene scale per hallway
- replace the pseudo-lux supervision with real photometric labels
- use a lightweight person detector or floor-occlusion segmenter on the Raspberry Pi
- log actual fixture wattage and regional grid factor instead of using defaults
- add temporal smoothing across consecutive accepted frames
- add a real deployment script that captures, filters, infers, exports CSV, and syncs results automatically


## 20. What You Still Need To Provide

The notebook already contains the Drive paths you gave me. To make the prototype as accurate as possible, the main remaining inputs are:
- a representative hallway floor area in square meters
- better estimates for fixture efficacy, utilization factor, and maintenance factor
- your local or regional grid emission factor
- a real hallway image for the single-image inference demo
- eventually, direct hallway lux measurements if you want this to move beyond prototype quality
